# BigQuant 量化大赛 · 端到端赛道 · 单 notebook 提交

模型包: `cnn_3f_1024_fuse_hs` —— **自有三频 CNN+Transformer 与团队高分双频模型的推理端融合**。

两路信号逐截面 rank 相关仅 ~0.52 (同族高分模型之间为 0.83), 说明二者捕捉的
alpha 来源不同, 融合是真正的互补。实测在 2024-07~12 六个完全独立于自有模型
训练区间的月份上 **6/6 全为正增益, 平均 IC +11.6%**。

融合后叠加**日内因果 L1 软阈值**降换手: 换手指标 `mean|Δpct_rank|` 是 L1 型
度量, 因此用同型软阈值而非 EMA(L2); 等换手预算下 IC_IR 损失约为 EMA 的一半。

本 notebook **自包含**: 内含超参数配置 / 随机种子 / 自有模型定义与数据管线 /
训练函数 `train_and_save` / 高分模型推理段 / 融合入口 `main`。仅额外依赖同目录的:

- `transformer_model.json` —— 已训练权重 (ours + hs 合并, 含标准化统计与结构超参)
- `requirements.txt` —— 依赖声明

平台约定:
- **公榜**: 调用 `main(datasources, start_date, end_date)`, 加载提交权重推理;
- **私榜**: 调用 `train_and_save(datasources)` 在隔离环境从零重训 (种子已固定)。


In [ ]:
import types
# ======================================================================
# 超参数配置 + 随机种子 (单一事实来源)
# ======================================================================
CFG = types.SimpleNamespace(
    SEED=42,
    FREQS=['bar30m', 'bar15m', 'bar5m'],
    LOOKBACK_MINUTES=120,
    HORIZON_MINUTES=30,
    SAMPLE_STEP_MINUTES=30,
    INPUT_SAME_DAY_ONLY=False,
    TRAIN_MONTHS=['202001', '202002', '202003', '202004', '202005', '202006', '202007', '202008', '202009', '202010', '202011', '202012', '202101', '202102', '202103', '202104', '202105', '202106', '202107', '202108', '202109', '202110', '202111', '202112', '202201', '202202', '202203', '202204', '202205', '202206', '202207', '202208', '202209', '202210', '202211', '202212', '202301', '202302', '202303', '202304', '202305', '202306', '202307', '202308', '202309', '202310', '202311', '202312'],
    MODEL_CFG={'in_dim': 25, 'hidden': 64, 'conv_blocks': 3, 'kernel': 3, 'transformer_layers': 2, 'heads': 4, 'dropout': 0.2, 'fused_dim': 64},
    TRAIN_CFG={'epochs': 12, 'lr': 0.001, 'weight_decay': 1e-05, 'batch_size': 4096, 'patience': 4, 'grad_clip': 1.0, 'val_months': 2, 'lr_factor': 0.5, 'lr_patience': 2, 'loss': 'corr', 'target': 'rank'},
    USE_XSEC=False,
    XSEC_LAYERS=1,
    XSEC_HEADS=4,
    USE_TURNOVER=True,
    LAMBDA_TURNOVER=0.005,
    USE_AMP=True,
    AMP_DTYPE='float16',
    EMA_ALPHA=0.45,
)


In [ ]:
"""
# Transformer 端到端 —— 训练侧 (单一事实来源)

本文件同时承担两件事, 是训练与推理的**唯一事实来源**:

1. **本地阶段**: 在写死的训练区间上从零训练 CNN+Transformer, 把
   **权重 + 标准化统计 + 结构超参** 存到 `transformer_model.json` (纯文本 JSON,
   张量以 dtype/shape/list 形式序列化, 便于平台原样上传)。
2. **私榜重训阶段**: 平台在隔离环境调用 `train_and_save(datasources)` 从零重训,
   用于验证提交不含违规(未来数据等)。

推理脚本 `Transformer_modelsave_predict.py` 从本文件 import 共享定义, 不重复实现,
保证 train / infer 预处理**严格一致**, 杜绝 train-infer 漂移与数据泄漏。

## 关键设计

- **只依赖 1 分钟 K 线** (`cpt_jyc_2026_e2e_bar1m`)。比赛平台保证提供 1/5/15/30 分钟
  四种数据, 且实测 **bar1m 按 5/15/30 分钟聚合可逐位复现原生 bar5m/15m/30m**
  (100% 数值一致), 因此推理时只在内部做重采样, 提交物自包含、无需外部多频数据。
- **CNN 滑窗 + Transformer**: 1x1 Conv 投影 -> 多个膨胀卷积残差块 -> 正弦位置编码
  -> TransformerEncoder -> 注意力池化‖平均池化 -> MLP head。
- **多频融合**: 每个频率一条独立编码分支, 池化后拼接, 残差式 head 输出标量。
- **标准化统计随权重落盘**: 训练集 fit, 推理直接复用。
- 不做任何手工因子(滚动统计/量价组合/盘口不平衡), 只做单位还原、量纲统一、
  对数变换与缺失值处理, 符合"基础算子"边界。
"""
import gc
import json
import math
import os

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F



In [ ]:
# ==========================================================================
# 常量
# ==========================================================================
import config as CFG            # 超参数 + 随机种子 (单一事实来源)

MODEL_PATH = "transformer_model.json"
BATCH = CFG.TRAIN_CFG["batch_size"]
SEED = CFG.SEED

FREQ_MINUTES = {"bar1m": 1, "bar5m": 5, "bar15m": 15, "bar30m": 30}
MORNING_START, MORNING_END = 570, 690      # 09:30, 11:30 (分钟)
AFTERNOON_START, AFTERNOON_END = 780, 900  # 13:00, 15:00
AUCTION_MINUTE = 565                       # 09:25 集合竞价截面 (=9*60+25)
SAMPLE_STEP = CFG.SAMPLE_STEP_MINUTES      # 采样网格(分钟)
HORIZON_MINUTES = CFG.HORIZON_MINUTES      # 预测未来 30 分钟 VWAP

# 比赛要求的日内截面 (平台契约 8 个: 09:30, 10:00, 10:30, 11:00, 11:30,
# 13:30, 14:00, 14:30)。与权威样例 submissions/baseline 的
# LEGAL_OUTPUT_HHMM={930,1000,...,1430} 对齐。
#   * 09:30 开盘截面: bar1m 无结束于 09:30 的 bar(只有 09:25 竞价 + 09:31 起),
#     故**内部锚点**仍用 09:25 竞价 bar(AUCTION_MINUTE=565) 作窗口末位(数据严格
#     早于 09:30), 仅在**输出时间戳**时把 565 对齐到 OPENING_SECTION=570(09:30)。
#   * 15:00(900) 无未来窗口(标签 NaN), 平台不将其列为截面, 故**不产出**(见
#     make_anchors 里的 AFTERNOON_END 过滤)。
OPENING_SECTION = 570                       # 09:30 开盘截面(输出时刻, 分钟)
ALIGNED_MINUTES = [AUCTION_MINUTE] + [      # 内部锚点分钟(09:25 竞价 bar 为开盘锚)
    m for m in range(MORNING_START + SAMPLE_STEP, MORNING_END + 1, SAMPLE_STEP)
] + [
    m for m in range(AFTERNOON_START + SAMPLE_STEP, AFTERNOON_END, SAMPLE_STEP)
]


def _trim():
    """把 glibc 空闲内存(main arena)归还操作系统, 压低 RSS 峰值。

    多频重采样/对齐会 malloc 大量临时缓冲, glibc 默认把释放的 chunk 留在
    arena 里备用; 逐月累积时这些保留内存会叠加成 RSS 尖峰。显式 trim 可在
    每个月的临时数据 `del` + `gc.collect()` 之后真正把内存交还内核, 让
    私榜重训(默认 48 个月, 平台限制 **30GB**)与公榜/评测(限制 **51200MB**)
    都留出充足余量。非 glibc 平台静默跳过 (import 失败即忽略)。
    """
    try:
        import ctypes
        ctypes.CDLL("libc.so.6").malloc_trim(0)
    except Exception:
        pass


def _concat_inplace(arrs):
    """把若干同宽数组拼成一个**新**矩阵, 逐块拷贝后立即释放源块。

    相比 `np.concatenate(arrs)`(拼接瞬间源块与结果**同时**驻留, 峰值 ≈ 2×),
    这里先分配结果, 再把每块**就地**拷入并随即把源块置空, 峰值只比结果多一个
    "最小块"。逐月累积 48 个月的特征矩阵时, 这一处能省下接近一整份特征矩阵
    (数 GB) 的瞬时尖峰。仅用于 2D 特征矩阵 (y/keys 极小, 仍走原逻辑)。
    """
    arrs = [a for a in arrs if a is not None and len(a)]
    if not arrs:
        return None
    if len(arrs) == 1:
        return arrs[0]
    out = np.empty((sum(len(a) for a in arrs),) + arrs[0].shape[1:],
                   dtype=arrs[0].dtype)
    pos = 0
    for i, a in enumerate(arrs):
        out[pos:pos + len(a)] = a
        pos += len(a)
        arrs[i] = None
    return out


# 需要的原始列(e2e 表 schema)
_PRICE_FIELDS = ["open", "high", "low", "close", "amount",
                 "ask_price1", "ask_price2", "ask_price3",
                 "bid_price1", "bid_price2", "bid_price3"]
_OHLC = ["open", "high", "low", "close"]
_BOOK = [f"{s}_{k}{i}" for s in ("ask", "bid")
         for k in ("price", "volume", "num_orders") for i in (1, 2, 3)]

# 25 个特征(7 K线 + 18 盘口)
KLINE_FEATURES = ["open_rel", "high_rel", "low_rel", "close_ret",
                  "log_volume", "log_amount", "log_deal_number"]
ORDERBOOK_FEATURES = (
    [f"{s}_price{i}_rel" for s in ("ask", "bid") for i in (1, 2, 3)]
    + [f"log_{s}_volume{i}" for s in ("ask", "bid") for i in (1, 2, 3)]
    + [f"log_{s}_orders{i}" for s in ("ask", "bid") for i in (1, 2, 3)]
)
ALL_FEATURES = KLINE_FEATURES + ORDERBOOK_FEATURES
N_FEATURES = len(ALL_FEATURES)  # 25

# ==========================================================================
# 内存策略 (平台硬限制: 公榜/评测 51200MB, 私榜重训 30GB, 超限强制终止)
# --------------------------------------------------------------------------
# 1. **只 SELECT 需要的列**, 并尽早降为 float32, 丢掉 adjust_factor 等无关列;
# 2. **按自然月分块读取**, 每块处理完立即释放原始 DataFrame (del + gc + _trim);
# 3. **不物化 [N,T,25] 滑窗矩阵**, 只保留 "特征矩阵 + 采样锚点下标",
#    窗口在喂模型时按 batch 现场 gather (内存 O(batch*T*F) ~ 50MB);
# 4. 标准化是逐列仿射运算, 作用在**特征矩阵**上一次即可,
#    与作用在每个窗口上完全等价, 但内存降为 1/T;
# 5. 每个分块前后各多读 PAD_DAYS 天, 保证跨月/跨日的输入窗口与
#    标签窗口都能回看/前瞻到真实 bar (锚点仍严格限制在本块区间内, 不重复);
# 6. 逐月累积出的特征矩阵最后用 `_concat_inplace` 合并 (峰值只比结果多一小块),
#    取代 `np.concatenate`(拼接瞬间 ≈ 2× 结果) 的瞬时尖峰;
# 7. 每月末调用 `_trim()` 把 glibc 空闲 arena 归还内核, 大幅压低常驻 RSS。
# ==========================================================================
PAD_DAYS = 10          # 分块读取时向前/向后额外缓冲的天数
_SCALE_COLS = (_OHLC + ["amount", "deal_number"]
               + [f"{s}_price{i}" for s in ("ask", "bid") for i in (1, 2, 3)])

# 从表里真正要取的列(裁剪掉 adjust_factor 等)
NEEDED_COLS = ["date", "instrument_id", "volume"] + _SCALE_COLS + _BOOK
NEEDED_COLS = list(dict.fromkeys(NEEDED_COLS))




In [ ]:
# ==========================================================================
# 模型 (与本地训练代码结构逐位一致)
# ==========================================================================
class PositionalEncoding(nn.Module):
    """正弦位置编码(非学习式, 泛化到任意长度)。"""

    def __init__(self, dim: int, max_len: int = 500):
        super().__init__()
        pe = torch.zeros(max_len, dim)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, dim, 2).float() * (-math.log(10000.0) / dim))
        pe[:, 0::2] = torch.sin(position * div)
        pe[:, 1::2] = torch.cos(position * div)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]


class ConvResBlock(nn.Module):
    """1D 卷积残差块 (Pre-LN + GELU)。输入/输出 [B, dim, T]。"""

    def __init__(self, dim, kernel_size=3, dilation=1, dropout=0.1):
        super().__init__()
        pad = (kernel_size - 1) // 2 * dilation
        self.conv1 = nn.Conv1d(dim, dim, kernel_size, padding=pad, dilation=dilation)
        self.conv2 = nn.Conv1d(dim, dim, kernel_size, padding=pad, dilation=dilation)
        self.norm1 = nn.LayerNorm(dim)
        self.norm2 = nn.LayerNorm(dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        residual = x
        x = self.norm1(x.transpose(1, 2)).transpose(1, 2)
        x = F.gelu(x)
        x = self.conv1(x)
        x = self.norm2(x.transpose(1, 2)).transpose(1, 2)
        x = F.gelu(x)
        x = self.dropout(x)
        x = self.conv2(x)
        return x + residual


class AttentionPooling(nn.Module):
    """注意力池化 + 平均池化拼接: [B,T,D] -> [B,2D]。"""

    def __init__(self, dim):
        super().__init__()
        h = max(dim // 2, 8)
        self.score = nn.Sequential(nn.Linear(dim, h), nn.Tanh(), nn.Linear(h, 1))
        self.out_dim = dim * 2

    def forward(self, x):
        w = torch.softmax(self.score(x), dim=1)
        att = (x * w).sum(dim=1)
        avg = x.mean(dim=1)
        return torch.cat([att, avg], dim=-1)


class CNNTransformerEncoder(nn.Module):
    """CNN + Transformer 编码器: [B,T,in_dim] -> [B,T,hidden]。"""

    def __init__(self, in_dim=25, hidden_dim=64, conv_blocks=3, kernel_size=3,
                 dilation_growth=1, transformer_layers=2, n_heads=4, dropout=0.1):
        super().__init__()
        self.stem = nn.Conv1d(in_dim, hidden_dim, 1)
        dilations = [dilation_growth ** i for i in range(conv_blocks)]
        self.conv_blocks = nn.ModuleList(
            [ConvResBlock(hidden_dim, kernel_size, d, dropout) for d in dilations])
        self.pos_enc = PositionalEncoding(hidden_dim)
        layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim, nhead=n_heads, dim_feedforward=hidden_dim * 4,
            dropout=dropout, activation="gelu", batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(layer, num_layers=transformer_layers)
        self.out_norm = nn.LayerNorm(hidden_dim)

    def forward(self, x):
        x = x.transpose(1, 2)
        x = self.stem(x)
        for block in self.conv_blocks:
            x = block(x)
        x = x.transpose(1, 2)
        x = self.pos_enc(x)
        x = self.transformer(x)
        return self.out_norm(x)


class CrossSectionAttention(nn.Module):
    """截面自注意力: 同截面内一只股票 attend 到其余股票。

    输入 [N, D]（一个大 batch 的融合表示），可选 sec_ids [N]（全局截面编号）。
    当给定 sec_ids 时用 block-diagonal mask 限制注意力只在同截面内, 使一个
    batch 可同时包含多个截面（随机大 batch 下也保持截面语义正确）。

    对 N 维置换等变, 不引入 per-instrument embedding, 避免记住股票身份、
    在测试集新股票上泛化变差。
    """

    def __init__(self, dim, heads=4, layers=1, dropout=0.1):
        super().__init__()
        self.blocks = nn.ModuleList()
        for _ in range(layers):
            self.blocks.append(nn.ModuleDict({
                "attn": nn.MultiheadAttention(dim, heads, dropout=dropout,
                                              batch_first=True),
                "norm1": nn.LayerNorm(dim),
                "ffn": nn.Sequential(nn.Linear(dim, dim * 4), nn.GELU(),
                                     nn.Linear(dim * 4, dim)),
                "norm2": nn.LayerNorm(dim),
            }))

    @staticmethod
    def _section_mask(sec_ids, device):
        """[N, N] bool mask: True 表示允许 attend（同截面）。"""
        return (sec_ids[:, None] == sec_ids[None, :]).to(device)

    def forward(self, z, sec_ids=None):           # [N, D]
        mask = None
        if sec_ids is not None:
            mask = self._section_mask(sec_ids, z.device)
        z = z.unsqueeze(0)                         # [1, N, D]
        for block in self.blocks:
            a, _ = block["attn"](z, z, z, attn_mask=mask, need_weights=False)
            z = block["norm1"](z + a)
            z = block["norm2"](z + block["ffn"](z))
        return z.squeeze(0)                        # [N, D]


class StockTransformer(nn.Module):
    """单频模型: Encoder -> Pool -> (可选截面注意力) -> Head -> scalar。
    forward(x:[B,T,25] 或 list, sec_ids=None)->[B]。"""

    def __init__(self, in_dim=N_FEATURES, hidden=64, conv_blocks=3, kernel=3,
                 transformer_layers=2, heads=4, dropout=0.1, fused_dim=64,
                 use_xsec=False, xsec_layers=1, xsec_heads=4):
        super().__init__()
        self.encoder = CNNTransformerEncoder(
            in_dim, hidden, conv_blocks, kernel, 1, transformer_layers, heads, dropout)
        self.pool = AttentionPooling(hidden)
        self.use_xsec = use_xsec
        if use_xsec:
            self.xsec = CrossSectionAttention(self.pool.out_dim, heads=xsec_heads,
                                              layers=xsec_layers, dropout=dropout)
            head_in = self.pool.out_dim * 2
        else:
            self.xsec = None
            head_in = self.pool.out_dim
        self.head = nn.Sequential(
            nn.Dropout(dropout), nn.Linear(head_in, fused_dim),
            nn.ReLU(), nn.Dropout(dropout), nn.Linear(fused_dim, 1))

    def forward(self, x, sec_ids=None):
        # 兼容主入口统一传入 list(多频分支); 单频取第 0 项
        if isinstance(x, (list, tuple)):
            x = x[0]
        z = self.pool(self.encoder(x))
        if self.xsec is not None:
            z = torch.cat([z, self.xsec(z, sec_ids=sec_ids)], dim=-1)
        return self.head(z).squeeze(-1)


class StockTransformerMF(nn.Module):
    """多频模型: 每频一条 CNN+Transformer 分支, 拼接后残差 head;
    融合表示上可选截面交叉注意力。forward(xs, sec_ids=None)->[B]。"""

    def __init__(self, freqs, in_dim=N_FEATURES, hidden=64, conv_blocks=3, kernel=3,
                 transformer_layers=2, heads=4, dropout=0.1, fused_dim=64,
                 use_xsec=False, xsec_layers=1, xsec_heads=4):
        super().__init__()
        self.freqs = list(freqs)
        self.branch = nn.ModuleDict()
        self.pool = nn.ModuleDict()
        for f in self.freqs:
            self.branch[f] = CNNTransformerEncoder(
                in_dim, hidden, conv_blocks, kernel, 1, transformer_layers, heads, dropout)
            self.pool[f] = AttentionPooling(hidden)
        d = len(self.freqs) * 2 * hidden
        self.fuse = nn.Sequential(nn.Linear(d, fused_dim), nn.ReLU(),
                                  nn.LayerNorm(fused_dim))
        self.use_xsec = use_xsec
        if use_xsec:
            self.xsec = CrossSectionAttention(fused_dim, heads=xsec_heads,
                                              layers=xsec_layers, dropout=dropout)
            # 融合表示 z ‖ 截面上下文 z_ctx ‖ 多频原始 h
            head_in = fused_dim * 2 + d
        else:
            self.xsec = None
            head_in = fused_dim + d
        self.head = nn.Sequential(
            nn.Dropout(dropout), nn.Linear(head_in, fused_dim),
            nn.ReLU(), nn.Dropout(dropout), nn.Linear(fused_dim, 1))

    def forward(self, xs, sec_ids=None):
        parts = [self.pool[f](self.branch[f](xs[k])) for k, f in enumerate(self.freqs)]
        h = torch.cat(parts, dim=-1)            # [B, d]
        z = self.fuse(h)                        # [B, fused_dim]
        if self.xsec is not None:
            z = torch.cat([z, self.xsec(z, sec_ids=sec_ids), h], dim=-1)
        else:
            z = torch.cat([z, h], dim=-1)
        return self.head(z).squeeze(-1)


def build_model(model_cfg):
    """按 model_cfg 构造模型(arch='single'|'multi')。"""
    kw = {k: v for k, v in model_cfg.items() if k not in ("arch", "freqs")}
    # 优先用 model_cfg 里显式记录的开关 (checkpoint 自描述), 否则回落到 CFG。
    kw["use_xsec"] = bool(model_cfg.get("use_xsec", getattr(CFG, "USE_XSEC", False)))
    kw["xsec_layers"] = int(model_cfg.get("xsec_layers", getattr(CFG, "XSEC_LAYERS", 1)))
    kw["xsec_heads"] = int(model_cfg.get("xsec_heads", getattr(CFG, "XSEC_HEADS", 4)))
    if model_cfg["arch"] == "multi":
        return StockTransformerMF(freqs=model_cfg["freqs"], **kw)
    return StockTransformer(**kw)




In [ ]:
# ==========================================================================
# 数据: 读表 -> 还原 -> 重采样 -> 特征 -> 切窗
# ==========================================================================
def _pick_cols(df, start=None, end=None):
    """只保留训练/推理真正需要的列, 并按 [start-PAD, end+PAD] 过滤, 同时降 dtype。

    日期过滤对内存**至关重要**: 当 `table` 是已物化的 DataFrame (平台表对象 / 本地
    验证) 时, 不过滤就会让每个月都重复处理整段数据 (48 个月 × 全量)。
    用 `df.loc[mask, keep]` 一次完成 行过滤 + 列裁剪, 只产生**一份**拷贝。
    """
    ren = {}
    for c in df.columns:
        lc = c.lower()
        if lc in ("time", "datetime", "date"):
            ren[c] = "date"
        elif lc in ("instrument", "symbol", "stock", "code", "sid"):
            ren[c] = "instrument_id"
    if ren:
        df = df.rename(columns=ren)
    keep = [c for c in NEEDED_COLS if c in df.columns]
    if "date" not in keep or "instrument_id" not in keep:
        raise ValueError(f"表缺少必要列 date/instrument_id; 实际列={list(df.columns)}")

    mask = None
    if start is not None and end is not None and "date" in df.columns:
        dt = pd.to_datetime(df["date"])
        lo = pd.Timestamp(start) - pd.Timedelta(days=PAD_DAYS)
        hi = pd.Timestamp(end) + pd.Timedelta(days=PAD_DAYS)
        tz = getattr(dt.dtype, "tz", None)
        if tz is not None:
            lo, hi = lo.tz_localize(tz), hi.tz_localize(tz)
        mask = ((dt >= lo) & (dt <= hi)).to_numpy()
    df = df[keep] if mask is None else df.loc[mask, keep]
    return _shrink(df)


def _fetch_table(table, start, end, columns=None):
    """
    取数据为 DataFrame (**只取需要的列**, 且必须带日期过滤)。

    - DataFrame: 直接裁剪列 + 按 [start-PAD, end+PAD] 过滤;
    - 平台表对象: 经 to_pandas()/to_df()/df() 取出后同样裁剪/过滤;
    - 表名(str/可 f-string 化): 走 dai.query, **只 SELECT 需要的列 + 日期过滤**
      (平台会提供约 1 年前置数据, 不加过滤会把整表读入内存 -> OOM/超时)。

    注意: 调用方负责传**已经含缓冲**的 [start, end] (见 _month_windows)。
    """
    cols = list(columns or NEEDED_COLS)
    if isinstance(table, pd.DataFrame):
        return _pick_cols(table, start, end)
    for attr in ("to_pandas", "to_df"):
        if hasattr(table, attr):
            obj = getattr(table, attr)
            out = obj() if callable(obj) else obj
            return _pick_cols(out, start, end)
    if hasattr(table, "df"):
        obj = table.df
        out = obj() if callable(obj) else obj
        return _pick_cols(out, start, end)
    # 表名: 用 dai 查询, 只 SELECT 需要的列 + 日期过滤
    import dai
    buf = (pd.Timestamp(start) - pd.Timedelta(days=PAD_DAYS)).strftime("%Y-%m-%d")
    end_s = (pd.Timestamp(end) + pd.Timedelta(days=PAD_DAYS)).strftime(
        "%Y-%m-%d %H:%M:%S")
    # 平台表用字符串列 `instrument`(如 '000001.SZ'), 本地内部名是 instrument_id;
    # SELECT 时把内部名映射回平台列名(与 baseline 一致: instrument::string AS
    # instrument), 后续 _pick_cols 会再 rename 回 instrument_id 并做稳定 int 编码。
    sel_cols = ["instrument::string AS instrument" if c == "instrument_id"
                else c for c in cols]
    sel = ", ".join(sel_cols) if isinstance(table, str) else "*"
    sql = f"SELECT {sel} FROM {table}"
    out = dai.query(sql, filters={"date": [buf, end_s]}).df()
    return _pick_cols(out)      # SQL 侧已过滤, 不再重复过滤


# 向后兼容别名(旧调用名)
def _to_dataframe(table, start=None, end=None):
    return _fetch_table(table, start, end)


def _encode_instrument(s):
    """把股票标识统一成**稳定、与子集无关**的 int64 key。

    平台 bar1m 表与 cpt_jyc_2026_instruments 的 instrument 是**字符串代码**
    (如 '000001.SZ'), 本地 e2e 表的 instrument_id 是 **int16 枚举**:
      * 整数 -> 原样 int64 (本地枚举, 与本地伪 dai 的股票池 int 一致);
      * 字符串 -> 确定性 MD5 前缀散列成 60-bit 非负整数 (与子集/顺序无关,
        同一字符串永远得到同一码; 输出时对 cpt_jyc_2026_instruments 用同款
        函数即可还原 join)。
    绝不用 factorize: 它会随停牌/复牌的子集漂移, 破坏跨月/跨表 join。
    返回 int64 ndarray (可直接进 int64 keys 矩阵, 下游无需改动)。
    """
    import hashlib
    s = pd.Series(s)
    if pd.api.types.is_integer_dtype(s.dtype):
        return s.to_numpy(dtype="int64")
    def _code(x):
        return int(hashlib.md5(str(x).encode("utf-8")).hexdigest()[:15], 16)
    return s.map(_code).to_numpy(dtype="int64")


def _shrink(df):
    """把还原后的 DataFrame 压到最小内存占用 (float32 + 编码后 int64 标的)。"""
    if "instrument_id" in df.columns:
        df["instrument_id"] = _encode_instrument(df["instrument_id"])
    for c in df.columns:
        if c in ("date", "day", "minute_of_day", "instrument_id"):
            continue
        if pd.api.types.is_float_dtype(df[c]):
            df[c] = df[c].astype("float32")
    return df


def _month_windows(start, end):
    """把 [start, end] 按自然月切成若干块, 返回 [(first_day, last_day, blocks)]。

    只有落在 [start, end] 内的月份才产生数据块 (缓冲由 _fetch_table 负责),
    因此块与块之间不会重复采样, 避免跨月重复计数。
    """
    months = pd.period_range(pd.Timestamp(start).to_period("M"),
                             pd.Timestamp(end).to_period("M"), freq="M")
    return [(m.start_time, m.end_time) for m in months]


def _normalize(df):
    """归一列名 + 裁剪列 + 时间/dtype 规整 (只保留必需列, 降内存)。"""
    ren = {}
    for c in df.columns:
        lc = c.lower()
        if lc in ("time", "datetime", "date"):
            ren[c] = "date"
        elif lc in ("instrument", "symbol", "stock", "code", "sid"):
            ren[c] = "instrument_id"
    if ren:
        df = df.rename(columns=ren)
    keep = [c for c in NEEDED_COLS if c in df.columns]
    if "date" not in keep or "instrument_id" not in keep:
        raise ValueError(f"表缺少必要列 date/instrument_id; 实际列={list(df.columns)}")
    df = df[keep]
    df["date"] = pd.to_datetime(df["date"])
    df["instrument_id"] = _encode_instrument(df["instrument_id"])
    return df


def restore_units(df):
    """单位还原 + 缺失值语义 (兼容本地 e2e 与平台云端两种口径)。

    * **价格类字段**(OHLC / amount / 盘口价):
      - 本地 e2e 表是 int 的"分"(cents, 平台约定本地数据 float→int 压缩),
        需 /100 还原为"元"; 缺失价记法 OHLC=-1、盘口价=0(该侧无挂单)。
      - 平台云端表是 float 的"元"(无需 /100); OHLC 缺失为 NaN、盘口缺失为 0。
      统一按 dtype 判断: 整数 → 视为"分"并 /100; 浮点 → 已是"元"直接保留。
      (基线 baseline/exp009 亦声明「云端价格为元，无需 ÷100」。)
    * **volume / deal_number / 盘口量 / 笔数** 均不缩放(股数/笔数), 只转 float32。

    就地修改 (不 copy), 并把结果降为 float32 以节省内存。
    """
    for f in _OHLC:
        if f in df.columns:
            if pd.api.types.is_integer_dtype(df[f]):
                df[f] = df[f].where(df[f] != -1).astype("float32") / 100.0
            else:
                df[f] = df[f].astype("float32")
    for f in _PRICE_FIELDS:
        if f in df.columns and f not in _OHLC:
            if pd.api.types.is_integer_dtype(df[f]):
                df[f] = df[f].astype("float32") / 100.0
            else:
                df[f] = df[f].astype("float32")
    for f in _BOOK:
        if f in df.columns and f.endswith(("1", "2", "3")) and "price" in f:
            df[f] = df[f].where(df[f] != 0.0)
    for f in ("volume", "deal_number"):
        if f in df.columns:
            df[f] = df[f].astype("float32")
    for f in [c for c in _BOOK if "price" not in c]:
        if f in df.columns:
            df[f] = df[f].astype("float32")
    # dtype 已在 _pick_cols 里降过 (float32/int64), 这里无需再 _shrink 一次拷贝
    return df


def tag_time(df):
    """打时间标签: minute_of_day; 剔除 09:25 之前的盘前指示价 bar(保留 09:25)。

    就地修改, 只保留交易时段内的行 (09:25 之后) 以进一步压缩内存。
    """
    dt = df["date"]
    is_dt = pd.api.types.is_datetime64_any_dtype(dt)
    mod = ((dt.dt.hour * 60 + dt.dt.minute) if is_dt
           else np.floor(dt.to_numpy("int64") / 60) % 1440).astype("int16")
    df["minute_of_day"] = mod
    df["day"] = ((dt.dt.normalize().dt.tz_localize(None)) if is_dt
                 else pd.to_datetime(df["date"].astype("int64"), unit="m")
                 .dt.normalize())
    # 先补列再过滤, 避免在切片上赋值 (SettingWithCopyWarning / 隐式拷贝)
    return df[df["minute_of_day"] >= AUCTION_MINUTE]


def resample_bars(df1m, kmins):
    """
    把 1 分钟 bar 聚合成 kmins 分钟 bar (**低内存**)。

    规则: 09:25 集合竞价 bar 原样保留; 连续竞价段(09:30 起)按
    `end = 09:30 + ceil((mod-09:30)/K)*K` 分桶, OHLC=首/高/低/末,
    volume/amount/deal_number=求和, 盘口字段=末值(快照)。
    已实测与原生 bar5m/bar15m/bar30m 数值逐位一致。

    内存策略 (旧实现峰值 ~4×输入, 是主要 OOM 来源):
      * 排序: 数据本身已按 (instrument, 时间) 有序 -> 先探测, 有序则**零拷贝**;
      * 不再切出 auction / cont 两份大拷贝, 而是用**分组键数组**直接 groupby:
        - auction 行 (09:25) 的桶键置 -1, 聚合后按桶键 >=0 过滤丢弃;
        - 连续竞价行的桶键用纯 numpy 定点计算 (不产生中间 Series);
      * 不往 df 里插临时列 (避免触发整表 copy), 最终只 concat 出 ~20 万行的结果。

    关键: 必须按 (instrument_id, 时间) 有序后再取 first/last, 否则
    `open`/`close`/盘口快照 会因输入未按时间排序而取错。
    """
    inst = df1m["instrument_id"].to_numpy()
    ti = df1m["minute_of_day"].to_numpy().astype("int64", copy=False)
    if len(inst) > 1:
        # 探测是否已按 (instrument asc, minute asc) 有序
        desc = (inst[1:] < inst[:-1]) | ((inst[1:] == inst[:-1]) & (ti[1:] < ti[:-1]))
        df = df1m if not bool(desc.any()) else df1m.sort_values(
            ["instrument_id", "date"], kind="mergesort")
    else:
        df = df1m
    if df is not df1m:
        inst = df["instrument_id"].to_numpy()
        ti = df["minute_of_day"].to_numpy().astype("int64", copy=False)
    if not pd.api.types.is_datetime64_any_dtype(df["day"]):
        df = df.assign(day=pd.to_datetime(df["day"]))
        inst = df["instrument_id"].to_numpy()
        ti = df["minute_of_day"].to_numpy().astype("int64", copy=False)

    cols = ["instrument_id", "minute_of_day", "day"] + list(_OHLC) + \
        ["volume", "amount", "deal_number"] + _BOOK
    cols = [c for c in cols if c in df.columns]

    is_auc = (ti == AUCTION_MINUTE)
    auction = df.loc[is_auc, cols].copy()          # 每标的每天仅 1 行, 极小

    # 桶键: 连续竞价 -> 09:30 + ceil((mod-09:30)/K)*K; 竞价行 -> -1
    # 注意必须用 ceil (与平台 bar Kmin 口径一致): bar 09:31 归入 09:35 桶,
    # 因此不会产生 "09:30" 这种空桶。
    rel = ti - MORNING_START
    bucket = np.empty(len(df), "int64")
    np.maximum(rel, 0, out=bucket)
    np.add(bucket, kmins - 1, out=bucket)
    np.floor_divide(bucket, kmins, out=bucket)
    np.multiply(bucket, kmins, out=bucket)
    np.add(bucket, MORNING_START, out=bucket)
    bucket[is_auc] = -1

    agg = {"open": ("open", "first"), "high": ("high", "max"),
           "low": ("low", "min"), "close": ("close", "last"),
           "volume": ("volume", "sum"), "amount": ("amount", "sum"),
           "deal_number": ("deal_number", "sum")}
    for c in _BOOK:
        agg[c] = (c, "last")

    # 分组键必须是 (instrument, day, bucket), 否则跨日/跨标的会串
    g = (df.groupby([inst, df["day"].to_numpy(), bucket], sort=False)
           .agg(**agg).reset_index())
    g.columns = ["instrument_id", "day", "minute_of_day"] + list(agg.keys())
    g = g[g["minute_of_day"] >= 0]                 # 丢弃集合竞价的分组

    out = pd.concat([auction, g[[c for c in cols if c in g.columns]]],
                    ignore_index=True)
    out["date"] = (pd.to_datetime(out["day"])
                   + pd.to_timedelta(out["minute_of_day"], unit="m"))
    return (out.dropna(subset=["day"])
               .sort_values(["instrument_id", "date"])
               .reset_index(drop=True))


def build_features(df):
    """构造 25 个特征(只做单位/量纲统一与对数变换), 返回 float32 矩阵。

    直接返回 **ndarray** 而非 DataFrame: 25 列 float32 是窗口 gather 的底层存储,
    省掉一遍 DataFrame 拷贝。
    """
    close = df["close"].to_numpy(dtype="float32", copy=False)
    valid = close > 0
    n = len(df)
    out = np.empty((n, N_FEATURES), dtype="float32")

    def put(name, vals):
        out[:, ALL_FEATURES.index(name)] = vals

    def rel(col):
        v = df[col].to_numpy(dtype="float32", copy=False)
        return np.where(valid, v / np.where(valid, close, 1.0) - 1.0, np.nan)

    put("open_rel", rel("open"))
    put("high_rel", rel("high"))
    put("low_rel", rel("low"))
    # close_ret: 同一 (instrument, day) 内上一根 bar 的 close
    prev = df.groupby(["instrument_id", "day"], sort=False)["close"].shift(1)
    prev = prev.to_numpy(dtype="float32", copy=False)
    put("close_ret", np.where(prev > 0, close / np.where(prev > 0, prev, 1.0) - 1.0,
                              np.nan))
    for name, col in (("log_volume", "volume"), ("log_amount", "amount"),
                      ("log_deal_number", "deal_number")):
        put(name, np.log1p(np.clip(df[col].to_numpy(dtype="float32", copy=False),
                                   0.0, None)))
    for i in (1, 2, 3):
        for s in ("ask", "bid"):
            put(f"{s}_price{i}_rel", rel(f"{s}_price{i}"))
            put(f"log_{s}_volume{i}",
                np.log1p(np.clip(df[f"{s}_volume{i}"].to_numpy("float32", copy=False),
                                 0.0, None)))
            put(f"log_{s}_orders{i}",
                np.log1p(np.clip(
                    df[f"{s}_num_orders{i}"].to_numpy("float32", copy=False),
                    0.0, None)))
    return out


def build_labels(df, horizon_bars):
    """标签 = end_price / future_vwap - 1, 窗口不跨交易日 (float32)。

    与平台 vwap_return 定义完全一致: future_vwap 为未来窗口的 VWAP,
    end_price 为未来窗口最后一根 bar 的收盘价 (close[t+h]), 则
        label = end_price / future_vwap - 1
    即"相对未来 VWAP 的收盘价溢价"。训练与推理方向统一, 无需取负/取反纠正。
    """
    grp = [df["instrument_id"], df["day"]]
    amt64 = df["amount"].astype("float64")
    vol64 = df["volume"].astype("float64")
    c_amt = amt64.groupby(grp, sort=False).cumsum()
    c_vol = vol64.groupby(grp, sort=False).cumsum()
    fwd_amt = c_amt.groupby(grp, sort=False).shift(-horizon_bars) - c_amt
    fwd_vol = c_vol.groupby(grp, sort=False).shift(-horizon_bars) - c_vol
    fwd_vwap = np.where(fwd_vol > 0, fwd_amt / fwd_vol, np.nan)
    end = df["close"].astype("float64").groupby(grp, sort=False).shift(-horizon_bars)
    end_a = end.to_numpy(dtype="float64")
    lab = np.where((fwd_vwap > 0) & (end_a > 0),
                   end_a / np.where(fwd_vwap > 0, fwd_vwap, 1.0) - 1.0, np.nan)
    return lab.astype("float32")


def make_anchors(feat, label, df, seq_len, minutes, with_label=True):
    """**不物化窗口**: 只返回采样锚点下标 + 元信息。

    窗口在喂模型时用 `gather_windows(feat, sel, seq_len)` 现场 gather,
    因此内存从 O(N·T·F) 降到 O(N)。anchor 语义与旧 `make_samples` 完全一致:
    采样点按**时钟分钟**对齐; 输入窗口只要求**同一只股票**(允许回溯到前一日,
    与训练口径 input_same_day_only=False 一致); 标签窗口仍严格当日。
    """
    n = len(df)
    if n <= seq_len:
        return (np.zeros((0,), "int64"), np.zeros((0,), "int64"),
                np.zeros((0,), "int64"), np.zeros((0, 3), "int64"))
    inst = df["instrument_id"].to_numpy()
    mod = df["minute_of_day"].to_numpy()
    day = pd.to_datetime(df["day"]).to_numpy().astype("datetime64[D]").astype("int64")

    i = np.arange(seq_len, n)
    keep = np.isin(mod[i], np.asarray(sorted(set(minutes)), dtype=mod.dtype))
    keep &= (inst[i] == inst[i - seq_len])   # 输入窗口不跨股票(允许跨日回溯)
    if with_label:
        keep &= ~np.isnan(label[i])
    sel = i[keep]
    if sel.size == 0:
        return (np.zeros((0,), "int64"), np.zeros((0,), "int64"),
                np.zeros((0,), "int64"), np.zeros((0, 3), "int64"))
    y = label[sel].astype("float32") if with_label else np.zeros(len(sel), "float32")
    keys = np.stack([inst[sel].astype("int64"), day[sel].astype("int64"),
                     mod[sel].astype("int64")], axis=1)
    return sel.astype("int64"), y, keys


def gather_windows(feat, sel, seq_len, out=None):
    """把锚点 sel 的 [T, F] 输入窗口 gather 成 [N, T, F] (仅在喂模型时调用)。

    等价于旧 `feat[sel[:,None] + arange(-T,0)]`, 但可按 batch 调用 -> 内存 O(batch·T·F)。
    """
    n = len(sel)
    if n == 0:
        return np.zeros((0, seq_len, N_FEATURES), "float32")
    if out is None:
        out = np.empty((n, seq_len, N_FEATURES), dtype="float32")
    offsets = seq_len - 1
    for j in range(seq_len):                      # 按时间步 gather, 避免造 idx 矩阵
        out[:, j, :] = feat[sel - offsets + j]
    return out


def _scale_feat(feat, mean, std):
    """在**特征矩阵**上做标准化 (与逐窗口标准化完全等价, 内存降为 1/T)。"""
    np.subtract(feat, np.asarray(mean, "float32").reshape(1, -1), out=feat)
    np.divide(feat, np.asarray(std, "float32").reshape(1, -1), out=feat)
    np.nan_to_num(feat, copy=False, nan=0.0, posinf=0.0, neginf=0.0)
    return feat


def _make_freq(df, freq, seq_len, minutes, with_label=True, stats=None):
    """单频: 重采样 -> 特征 -> (可选)标签 -> 采样锚点; 就地标准化并释放中间量。

    返回 dict: keys, y(或 None), anchors, feat(已标准化), sort_idx (按 keys 排序)。
    """
    km = FREQ_MINUTES[freq]
    bars = resample_bars(df, km)
    feat = build_features(bars)
    lab = (build_labels(bars, max(1, HORIZON_MINUTES // km)) if with_label
           else np.zeros(len(bars), "float32"))
    sel, y, keys = make_anchors(feat, lab, bars.reset_index(drop=True),
                                seq_len, minutes, with_label)
    del lab
    if stats is not None:
        feat = _scale_feat(feat, stats[0], stats[1])
    order = np.lexsort((keys[:, 2], keys[:, 1], keys[:, 0])) if len(keys) else \
        np.zeros(0, "int64")
    return {"freq": freq, "seq_len": seq_len, "keys": keys, "y": y,
            "anchors": sel, "feat": feat, "order": order,
            "n_bars": len(bars)}


def _subset_freq(fd, sel):
    """按行下标取单频子集 (用于把采样时刻限制在测试区间)。"""
    o = dict(fd)
    o["keys"] = fd["keys"][sel]
    o["y"] = None if fd["y"] is None else fd["y"][sel]
    o["anchors"] = fd["anchors"][sel]
    return o


def _align_freqs(per, freqs):
    """多频: 按 keys 取交集(严格同一决策时刻), 返回 {freq: 行下标}。"""
    if len(freqs) == 1:
        return {freqs[0]: np.arange(len(per[freqs[0]]["keys"]))}
    common = set.intersection(*[set(map(tuple, per[f]["keys"].tolist()))
                                for f in freqs])
    common = set(sorted(common))
    out = {}
    for f in freqs:
        k = per[f]["keys"]
        mask = np.fromiter((tuple(r) in common for r in k.tolist()),
                           dtype=bool, count=len(k))
        out[f] = np.where(mask)[0]
    return out


def _month_span(month):
    """把 'YYYYMM' 转成 (月首时间戳, 月末时间戳)。"""
    p = pd.Period(str(month), freq="M")
    return p.start_time, p.end_time


def _empty_month(freqs, seq_lens):
    return ([np.zeros((0, int(seq_lens.get(f, 1)), N_FEATURES), "float32")
             for f in freqs],
            np.zeros(0, "float32"),
            pd.DataFrame({"date": [], "instrument": [], "minute_of_day": []}),
            np.zeros((0, 3), "int64"))


def _build_month(table, first, last, mode, minutes, stats, freqs, seq_lens):
    """构造**单个月**的模型输入 (原始 1 分钟表用后即弃 -> 内存峰值 ~1 个月)。

    返回 (X_list, y, idx_df, keys); 锚点严格限制在 [first, last] 内,
    因此按月调用不会跨月重复采样。stats 为 dict{freq: (mean, std)}。
    """
    with_label = (mode != "infer")
    raw = _normalize(_fetch_table(table, first, last))
    if len(raw) == 0:
        return _empty_month(freqs, seq_lens)
    raw = restore_units(raw)
    raw = tag_time(raw)

    per = {}
    for f in freqs:
        T = int(seq_lens.get(f, max(1, CFG.LOOKBACK_MINUTES // FREQ_MINUTES[f])))
        per[f] = _make_freq(raw, f, T, minutes, with_label=with_label,
                            stats=(stats or {}).get(f))
    # 把锚点限制在这一个月内 (避免缓冲期重复采样)
    d0 = np.datetime64(pd.Timestamp(first).date(), "D").astype("int64")
    d1 = np.datetime64(pd.Timestamp(last).date(), "D").astype("int64")
    for f in freqs:
        k = per[f]["keys"]
        if len(k):
            m = (k[:, 1] >= d0) & (k[:, 1] <= d1)
            per[f] = _subset_freq(per[f], np.where(m)[0])

    # 多频按 keys 取交集(严格同一决策时刻)
    idx = _align_freqs(per, freqs)
    ref = per[freqs[0]]
    j0 = idx[freqs[0]]
    keys = ref["keys"][j0]
    y = (ref["y"][j0] if ref["y"] is not None
         else np.zeros(len(keys), "float32")).astype("float32")

    X_list = []
    for f in freqs:
        T = int(seq_lens.get(f, max(1, CFG.LOOKBACK_MINUTES // FREQ_MINUTES[f])))
        X_list.append(gather_windows(per[f]["feat"], per[f]["anchors"][idx[f]], T))
    if len(keys):
        # 输出时间戳对齐平台 8 截面契约 (baseline LEGAL_OUTPUT_HHMM):
        #   09:25 竞价锚点 -> 输出 09:30 (开盘截面; bar1m 无 09:30 结束的 bar)
        #   其余整/半点原样; 15:00 已被 make_anchors 剔除(无未来窗口, 平台不列)
        out_mod = keys[:, 2].copy()
        out_mod[out_mod == AUCTION_MINUTE] = OPENING_SECTION
        idx_df = pd.DataFrame({
            "date": (pd.to_datetime(keys[:, 1], unit="D")
                     + pd.to_timedelta(out_mod, unit="m")),
            "instrument": keys[:, 0].astype("int64"),
            "minute_of_day": out_mod.astype("int64"),
        })
    else:
        idx_df = pd.DataFrame({"date": [], "instrument": [], "minute_of_day": []})
    del raw, per
    gc.collect()
    _trim()
    return X_list, y, idx_df, keys


def _norm_freqs(freqs, seq_lens, minutes):
    freqs = list(freqs or ["bar30m"])
    seq_lens = dict(seq_lens or {})
    return freqs, seq_lens, (ALIGNED_MINUTES if minutes is None else minutes)


def build_dataset(table, start_date, end_date, mode, minutes, stats,
                  freqs=None, seq_lens=None):
    """
    从 1 分钟表构造模型输入 (**分月流式, 内存受限**)。

    返回 (X_list, y, idx_df, keys):
      X_list : list[np.float32[N, T_f, 25]]  每个频率一项(单频即长度 1)
      y      : np.float32[N]  (infer 模式全 0)
      idx_df : DataFrame[date, instrument, minute_of_day]  预测截面键
      keys   : [N,3] (instrument_id, day_ordinal, minute_of_day)

    内存策略: 逐月调用 `_build_month`, 每个月只读 [月初-PAD, 月末+PAD] 的 bar1m,
    处理完立即释放; 采样锚点严格限制在该月内, 因此各块之间不重复。
    依赖长区间的推理请改用 `_iter_dataset` (逐月 yield, 峰值只有 1 个月)。
    stats  : dict{freq: (mean, std)}; 推理直接复用训练统计量。
    """
    freqs, seq_lens, minutes = _norm_freqs(freqs, seq_lens, minutes)
    Xs_acc = [[] for _ in freqs]
    ys, idxs, keyss = [], [], []
    for first, last in _month_windows(start_date, end_date):
        Xs, y, idx_df, keys = _build_month(
            table, first, last, mode, minutes, stats, freqs, seq_lens)
        if len(idx_df) == 0:
            continue
        for i in range(len(freqs)):
            Xs_acc[i].append(Xs[i])
        ys.append(y)
        idxs.append(idx_df)
        keyss.append(keys)
    if not idxs:
        return _empty_month(freqs, seq_lens)
    X_list = [np.concatenate(a) if a else
              np.zeros((0, int(seq_lens.get(f, 1)), N_FEATURES), "float32")
              for a, f in zip(Xs_acc, freqs)]
    idx_df = pd.concat(idxs, ignore_index=True)
    return X_list, np.concatenate(ys), idx_df, np.concatenate(keyss)


def _iter_dataset(table, start_date, end_date, mode, minutes, stats,
                  freqs=None, seq_lens=None):
    """**逐月 yield** 单月模型输入 (推理用)。

    每次 yield `(X_list, y, idx_df, keys)`, 调用方消费完即可释放 -> 峰值内存
    只有 1 个月的窗口 (~数百 MB), 长测试区间也不会触发平台 51200MB 限制。
    """
    freqs, seq_lens, minutes = _norm_freqs(freqs, seq_lens, minutes)
    for first, last in _month_windows(start_date, end_date):
        Xs, y, idx_df, keys = _build_month(
            table, first, last, mode, minutes, stats, freqs, seq_lens)
        if len(idx_df) == 0:
            continue
        yield Xs, y, idx_df, keys
        # 调用方已消费完本月张量 -> 显式释放并归还 glibc 空闲 arena,
        # 使长评测区间的常驻 RSS 稳定在"单月"量级 (与 _accumulate 同策略)。
        del Xs, y, idx_df, keys
        gc.collect()
        _trim()


def _fit_scaler(feat, chunk=1_000_000):
    """在**特征矩阵** [n_bars, F] 上算均值/标准差 (分块累加, 忽略 NaN)。

    注: 训练统计量沿用"在 bar 级特征矩阵上 fit"的口径 (与推理侧复用同一公式),
    标准化是逐列仿射, 作用在 feat 上与作用在窗口上结果一致, 但内存降为 1/T。
    """
    nf = feat.shape[-1]
    s = np.zeros(nf, "float64"); sq = np.zeros(nf, "float64"); c = np.zeros(nf, "float64")
    for st in range(0, len(feat), chunk):
        blk = feat[st:st + chunk].astype("float64")
        m = ~np.isnan(blk)
        z = np.where(m, blk, 0.0)
        s += z.sum(0); sq += (z * z).sum(0); c += m.sum(0)
    c = np.maximum(c, 1.0)
    mean = s / c
    var = np.maximum(sq / c - mean * mean, 0.0)
    std = np.sqrt(var)
    mean = np.where(np.isnan(mean), 0.0, mean)
    std = np.where((np.isnan(std)) | (std < 1e-6), 1.0, std)
    return mean.astype("float32"), std.astype("float32")


def _rank_targets(y, keys):
    """截面 rank 目标: 同一 (day, mod) 内做百分位排名并映射到 [-1,1]。"""
    out = np.zeros_like(y, "float32")
    df = pd.DataFrame({"y": y, "day": keys[:, 1], "mod": keys[:, 2]})
    for _, g in df.groupby(["day", "mod"], sort=False):
        if len(g) < 2:
            continue
        r = g["y"].rank(method="average")
        out[g.index.to_numpy()] = ((r / len(g)) * 2.0 - 1.0).to_numpy("float32")
    return out


def _corr_batch(p, y):
    """batch 内 Pearson 相关 (中心化后余弦), 返回标量 corr ∈ [-1, 1]。

    注意: eps 放在 sqrt **内部**, 否则当 p/y 均为常数 (方差=0) 时,
    sqrt(0) 的反向会得到 inf*0 = NaN 梯度 (如换手正则的单元素配对)。

    另: 前向在 autocast(fp16/bf16) 下 p/y 可能是低精度张量, 这里先 .float()
    升回 fp32, 保证 corr 与梯度的数值精度不受混合精度影响。
    """
    p = p.float(); y = y.float()
    pc = p - p.mean(); yc = y - y.mean()
    denom = torch.sqrt((pc ** 2).sum() * (yc ** 2).sum() + 1e-8)
    return (pc * yc).sum() / denom


def _grouped_corr(p, y, sidb):
    """按截面分组求 corr 再平均 (向量化, 替代逐 group 串行循环)。

    用 scatter_add 一次归约所有 group 的 sum/count/sumsq, 与逐 group 的
    `_corr_batch` 数值及梯度一致 (对拍误差 < 1e-8)。有效 group (<2 样本)
    不计入, 全部无效时退化为整 batch corr。返回标量均值。
    """
    p = p.float(); y = y.float()
    ids = torch.unique(sidb)
    gid = torch.searchsorted(ids, sidb)                 # 连续 id 0..G-1
    G = ids.numel()
    cnt = torch.bincount(gid, minlength=G).float()      # [G]
    sp = torch.zeros(G, device=p.device, dtype=p.dtype)
    sy = torch.zeros(G, device=p.device, dtype=p.dtype)
    sp.scatter_add_(0, gid, p)
    sy.scatter_add_(0, gid, y)
    mp = sp / cnt
    my = sy / cnt
    pc = p - mp[gid]
    yc = y - my[gid]
    ssp = torch.zeros(G, device=p.device, dtype=p.dtype)
    ssy = torch.zeros(G, device=p.device, dtype=p.dtype)
    sxy = torch.zeros(G, device=p.device, dtype=p.dtype)
    ssp.scatter_add_(0, gid, pc * pc)
    ssy.scatter_add_(0, gid, yc * yc)
    sxy.scatter_add_(0, gid, pc * yc)
    corrs = sxy / torch.sqrt(ssp * ssy + 1e-8)          # [G]
    valid = cnt >= 2
    if valid.sum() == 0:
        return _corr_batch(p, y)
    return corrs[valid].mean()


def _section_ids(keys):
    """把 (day, minute) 映射为截面编号 [N] int (0..K-1)。

    用于截面交叉注意力的 block-diagonal mask: 编号只参与相等比较,
    因此用 day*1440 + mod 的整数哈希即可, 无需额外的连续映射开销。
    返回 int64 数组, 随机 batch 下按全局下标切片即可还原截面身份。
    """
    if len(keys) == 0:
        return np.zeros(0, "int64")
    sec = keys[:, 1].astype("int64") * 1440 + keys[:, 2].astype("int64")
    _, ids = np.unique(sec, return_inverse=True)
    return ids.astype("int64")


def _turnover_next(keys):
    """返回 next_idx [N]: 样本 i 的下一个同 (instrument, day) 截面下标, 无则 -1。

    换手正则用: 相邻截面是"同一只股票在同一交易日的下一采样时刻"。用
    (instrument, day, mod) 的字典序定位同 (instrument, day) 块内 mod 升序的
    后继行。与输入 keys 是否已排序无关 (内部 lexsort 一次, 代价 O(N log N),
    整个训练周期仅算一次)。
    """
    n = len(keys)
    next_idx = np.full(n, -1, "int64")
    if n < 2:
        return next_idx
    order = np.lexsort((keys[:, 2], keys[:, 1], keys[:, 0]))
    k = keys[order]
    same = (k[1:, 0] == k[:-1, 0]) & (k[1:, 1] == k[:-1, 1])
    valid = same & (k[1:, 2] != k[:-1, 2])      # 同截面自身不配对
    nxt_sorted = np.full(n, -1, "int64")
    nxt_sorted[np.arange(0, n - 1)[valid]] = np.arange(1, n)[valid]
    inv = np.empty(n, "int64")
    inv[order] = np.arange(n)
    m = nxt_sorted >= 0
    next_idx[order[m]] = order[nxt_sorted[m]]
    return next_idx


def clip_bounds(labels, lo=0.01, hi=0.99):
    x = labels[np.isfinite(labels)]
    if x.size == 0:
        return -np.inf, np.inf
    a, b = np.quantile(x, [lo, hi])
    return float(a), float(b)




In [ ]:
# ==========================================================================
# 模型存取 (JSON: 张量按 dtype/shape/list 序列化)
# ==========================================================================
def _enc(obj):
    if isinstance(obj, dict):
        return {k: _enc(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_enc(v) for v in obj]
    if isinstance(obj, torch.Tensor):
        a = obj.detach().cpu().numpy()
        return {"__t__": 1, "dtype": str(a.dtype), "shape": list(a.shape),
                "data": a.reshape(-1).tolist()}
    if isinstance(obj, np.ndarray):
        return {"__t__": 1, "dtype": str(obj.dtype), "shape": list(obj.shape),
                "data": obj.reshape(-1).tolist()}
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.integer,)):
        return int(obj)
    return obj


def _dec(obj):
    if isinstance(obj, dict):
        if obj.get("__t__"):
            a = np.array(obj["data"], dtype=obj["dtype"]).reshape(obj["shape"])
            return a
        return {k: _dec(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_dec(v) for v in obj]
    return obj


def save_model(path, payload):
    with open(path, "w", encoding="utf-8") as fh:
        json.dump(_enc(payload), fh)


def load_model(path, map_location="cpu"):
    with open(path, "r", encoding="utf-8") as fh:
        raw = json.load(fh)
    ck = _dec(raw)
    if "state_dict" in ck:
        ck["state_dict"] = {k: torch.as_tensor(v, device=map_location)
                            for k, v in ck["state_dict"].items()}
    return ck




In [ ]:
# ==========================================================================
# 训练
# ==========================================================================
def _accumulate(table, months, freqs, seq_lens, minutes, stats=None):
    """**按月流式**累积训练张量: 只保留 "频率特征矩阵 + 锚点下标 + keys"。

    内存关键点(平台硬限制 51200MB):
      * 原始 bar1m 只在一个月的范围内驻留, 每月处理完立即 `del` + `gc.collect()`;
      * **不物化 [N, T, F] 滑窗矩阵**, 只存 `feat` (N_bars×25) 与锚点 (N,);
      * 多频严格按 (instrument, day, minute) 取交集, 与推理口径一致;
      * 各月锚点都限制在本月内, 因此块与块之间不会重复采样。

    返回 dict: feats{freq: [n_bars, 25] float32}, anchors{freq: [N] int64},
               y[N] float32, keys[N, 3]
    """
    acc = {"feats": {f: [] for f in freqs}, "anchors": {f: [] for f in freqs},
           "keys": [], "y": []}
    off = {f: 0 for f in freqs}
    for m in months:
        first, last = _month_span(m)
        raw = _normalize(_fetch_table(table, first, last))
        if len(raw) == 0:
            continue
        raw = restore_units(raw)
        raw = tag_time(raw)
        per = {}
        for f in freqs:
            per[f] = _make_freq(raw, f, int(seq_lens[f]), minutes,
                                with_label=True, stats=(stats or {}).get(f))
        # 锚点限制在本月内, 避免缓冲期重复采样
        d0 = np.datetime64(pd.Timestamp(first).date(), "D").astype("int64")
        d1 = np.datetime64(pd.Timestamp(last).date(), "D").astype("int64")
        for f in freqs:
            k = per[f]["keys"]
            if len(k):
                mask = (k[:, 1] >= d0) & (k[:, 1] <= d1)
                per[f] = _subset_freq(per[f], np.where(mask)[0])
        # 多频取交集 -> 同一决策时刻
        idx = _align_freqs(per, freqs)
        j0 = idx[freqs[0]]
        acc["keys"].append(per[freqs[0]]["keys"][j0])
        acc["y"].append(per[freqs[0]]["y"][j0])
        for f in freqs:
            acc["feats"][f].append(per[f]["feat"])
            # 锚点转为"拼接后特征矩阵"的全局行号
            acc["anchors"][f].append(per[f]["anchors"][idx[f]] + off[f])
            off[f] += len(per[f]["feat"])
        del raw, per
        gc.collect()
        _trim()

    out = {}
    if not acc["keys"]:
        out["keys"] = np.zeros((0, 3), "int64")
        out["y"] = np.zeros(0, "float32")
    else:
        out["keys"] = np.concatenate(acc["keys"])
        out["y"] = np.concatenate(acc["y"]).astype("float32")
    out["feats"] = {}
    for f in freqs:
        c = _concat_inplace(acc["feats"][f])
        out["feats"][f] = (c if c is not None
                           else np.zeros((0, N_FEATURES), "float32"))
    out["anchors"] = {f: (np.concatenate(acc["anchors"][f]) if acc["anchors"][f]
                          else np.zeros(0, "int64")) for f in freqs}
    del acc
    gc.collect()
    _trim()
    return out


def _batch_windows(feats, anchors, row_idx, seq_lens, freqs, buf=None):
    """按行下标现场 gather 一个 batch 的窗口 -> list[np.float32[b, T_f, 25]]。

    复用 `buf`(预分配) 以避免每个 step 反复 malloc, 峰值内存 O(batch·T·F) ~ 50MB。
    """
    out = []
    for f in freqs:
        sel = anchors[f][row_idx]
        if buf is not None and len(row_idx) == len(buf[f]):
            b = buf[f]
            gather_windows(feats[f], sel, int(seq_lens[f]), out=b)
        else:
            b = gather_windows(feats[f], sel, int(seq_lens[f]))
        out.append(b)
    return out


def _train_freqs(table, train_months, freqs, seq_lens, minutes, seed=SEED,
                 epochs=None, lr=None, batch=None, patience=None,
                 weight_decay=None, val_months=None, lam_turnover=None):
    """在给定 1 分钟表上训练多频模型, 返回 (model, stats, model_cfg)。

    数据侧**分月流式**构造 (见 `_accumulate`): 训练集与验证集各扫一遍自己的月份,
    全程不物化 [N, T, F] 滑窗矩阵, 窗口在训练 step 里按 batch 现场 gather。

    可复现性: 固定 Python / NumPy / PyTorch(含 CUDA) 随机种子, 并关闭 cudnn 的
    benchmark 自动调优, 使 CPU/GPU 结果尽量一致。
    """
    tc = CFG.TRAIN_CFG
    epochs = tc["epochs"] if epochs is None else epochs
    lr = tc["lr"] if lr is None else lr
    batch = tc["batch_size"] if batch is None else batch
    patience = tc["patience"] if patience is None else patience
    weight_decay = tc["weight_decay"] if weight_decay is None else weight_decay
    val_months = tc["val_months"] if val_months is None else val_months

    # ---- 固定随机种子 ----
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    # 切分 train/val: 末尾 val_months 个月做验证 (时间序列, 不打乱)
    months = sorted(str(m) for m in train_months)
    val_m = months[-val_months:] if len(months) > val_months + 1 else months[-1:]
    tr_m = [m for m in months if m not in set(val_m)]
    print(f"  [data] train={len(tr_m)}月 val={len(val_m)}月 ({val_m[0]}~{val_m[-1]})")

    # ---- 训练集: 直接 fit 统计量 (不物化窗口), 再就地标准化 ----
    tr = _accumulate(table, tr_m, freqs, seq_lens, minutes, stats=None)
    if len(tr["keys"]) == 0:
        raise RuntimeError(f"训练集无样本 (months={tr_m})")
    stats = {}
    for f in freqs:
        mean, std = _fit_scaler(tr["feats"][f])
        _scale_feat(tr["feats"][f], mean, std)
        stats[f] = (mean.astype("float32"), std.astype("float32"))

    # ---- 验证集: 复用训练统计量 (同口径 -> 无泄漏) ----
    va = _accumulate(table, val_m, freqs, seq_lens, minutes, stats=stats)
    if len(va["keys"]) == 0:
        raise RuntimeError(f"验证集无样本 (months={val_m})")
    print(f"  [data] train_N={len(tr['keys']):,} val_N={len(va['keys']):,}")

    ytr = _rank_targets(tr["y"], tr["keys"])
    yva_raw = va["y"]
    vkeys = va["keys"]

    model_cfg = dict(arch="multi" if len(freqs) > 1 else "single",
                     freqs=list(freqs), **CFG.MODEL_CFG,
                     use_xsec=bool(getattr(CFG, "USE_XSEC", False)),
                     xsec_layers=int(getattr(CFG, "XSEC_LAYERS", 1)),
                     xsec_heads=int(getattr(CFG, "XSEC_HEADS", 4)))
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = build_model(model_cfg).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt, factor=tc["lr_factor"], patience=tc["lr_patience"])

    # ---- 混合精度 (AMP) ----
    # float16 需要 GradScaler 防下溢; bfloat16 动态范围与 fp32 一致, 无需 scaler。
    # CPU / 无 CUDA 时自动关闭 (dtype 回退 fp32)。
    use_amp = bool(getattr(CFG, "USE_AMP", False)) and device.type == "cuda"
    amp_dtype = {"float16": torch.float16,
                 "bfloat16": torch.bfloat16}.get(
        str(getattr(CFG, "AMP_DTYPE", "float16")), torch.float16)
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp) if use_amp else None
    if use_amp:
        print(f"  [amp] enabled dtype={amp_dtype}")

    n = len(ytr)
    ytr_t = torch.from_numpy(ytr)
    buf = {f: np.empty((batch, int(seq_lens[f]), N_FEATURES), "float32")
           for f in freqs}
    grad_clip = float(tc.get("grad_clip", 1.0))

    use_xsec = bool(getattr(CFG, "USE_XSEC", False))
    use_turnover = bool(getattr(CFG, "USE_TURNOVER", False))
    lam_to = (float(getattr(CFG, "LAMBDA_TURNOVER", 0.01))
              if lam_turnover is None else float(lam_turnover))

    # 截面编号 (训练周期仅算一次)
    sec_ids_tr = _section_ids(tr["keys"]) if use_xsec else None

    # 换手"下一截面"配对 (全局下标, 仅算一次): cur -> 同 (instrument, day) 的
    # 下一采样截面。独立配对 batch: 每 epoch 采样一批 (cur, next) 对, 换手 loss
    # 100% 命中, 不受随机 batch 打散影响 (原随机 batch 命中率仅 ~1.4%)。
    cur_tr = nxt_tr = None
    if use_turnover:
        nxt_all = _turnover_next(tr["keys"])
        valid = nxt_all >= 0
        cur_tr = np.where(valid)[0].astype("int64")
        nxt_tr = nxt_all[valid].astype("int64")
        del nxt_all, valid
    n_pair = len(cur_tr) if cur_tr is not None else 0

    best, best_state, bad = -9e9, None, 0

    for ep in range(epochs):
        model.train()
        perm = torch.randperm(n)
        perm_np = perm.numpy()
        tot = 0.0; nb = 0
        tot_to = 0.0; cnt_to = 0

        # 换手独立配对 batch: 每 epoch 均匀采样 k 对 (cur, next) 作为额外的
        # 前向 step (不更新主损失方向, 只叠加换手正则)。命中率 100%。
        to_cur = to_nxt = None
        if use_turnover and n_pair >= 2:
            k = min(n_pair, batch // 2)
            sel = np.random.permutation(n_pair)[:k]
            to_cur = cur_tr[sel]
            to_nxt = nxt_tr[sel]

        for s in range(0, n, batch):
            idxb = perm_np[s:s + batch]
            xs = _batch_windows(tr["feats"], tr["anchors"], idxb, seq_lens,
                                freqs, buf=buf)
            xb = [torch.from_numpy(x).to(device) for x in xs]
            yb = ytr_t[idxb].to(device)
            sidb = None if sec_ids_tr is None else \
                torch.from_numpy(sec_ids_tr[idxb]).to(device)

            opt.zero_grad()
            # AMP: 前向 + 损失在 autocast 上下文中; 损失内部 _corr_batch 已 .float(),
            # 故 loss 数值与梯度精度不受低精度前向影响。
            with torch.autocast(device_type="cuda", dtype=amp_dtype,
                                enabled=use_amp):
                p = model(xb, sec_ids=sidb)

                # 主损失: 与评估截面 IC 对齐。启用截面注意力时改为"组内 corr 再平均",
                # 避免随机大 batch 把多个截面混在一起做 Pearson 造成的偏差。
                if use_xsec:
                    loss = 1.0 - _grouped_corr(p, yb, sidb)
                else:
                    loss = 1.0 - _corr_batch(p, yb)

            if scaler is not None:
                scaler.scale(loss).backward()
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                scaler.step(opt)
                scaler.update()
            else:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                opt.step()
            tot += float(loss); nb += 1

        # ---- 换手独立配对 step (100% 命中) ----
        # 采样一批 (cur, next) 配对做一次额外前向, 叠加 λ·(1 - corr(p_cur, p_next))。
        # 独立于主 batch, 不扰动主损失方向。
        if use_turnover and to_cur is not None and to_nxt is not None:
            xc = _batch_windows(tr["feats"], tr["anchors"], to_cur, seq_lens,
                                freqs, buf=None)
            xn = _batch_windows(tr["feats"], tr["anchors"], to_nxt, seq_lens,
                                freqs, buf=None)
            xcb = [torch.from_numpy(x).to(device) for x in xc]
            xnb = [torch.from_numpy(x).to(device) for x in xn]
            opt.zero_grad()
            with torch.autocast(device_type="cuda", dtype=amp_dtype,
                                enabled=use_amp):
                pc = model(xcb, sec_ids=None)
                pn = model(xnb, sec_ids=None)
                to_corr = _corr_batch(pc, pn)
                to_loss = lam_to * (1.0 - to_corr)
            if scaler is not None:
                scaler.scale(to_loss).backward()
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                scaler.step(opt)
                scaler.update()
            else:
                to_loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                opt.step()
            tot_to += float(1.0 - to_corr.detach()); cnt_to += 1

        ic = _eval_ic(model, va["feats"], va["anchors"], seq_lens,
                      yva_raw, vkeys, freqs, device)
        sched.step(-ic)
        msg = f"  ep{ep+1:02d} loss={tot/max(nb,1):.4f} val_IC={ic:+.4f}"
        if use_turnover:
            msg += f" turnover={tot_to/max(cnt_to,1):.4f}"
            vto = _eval_turnover(model, va["feats"], va["anchors"], seq_lens,
                                 freqs, vkeys, device)
            msg += f" val_turnover={vto:.4f}"
        print(msg)
        if ic > best:
            best, bad = ic, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= patience:
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    model.eval()
    print(f"  best val_IC={best:+.4f}")
    del tr, va
    gc.collect()
    return model, stats, model_cfg


@torch.no_grad()
def _predict_all(model, feats, anchors, seq_lens, freqs, keys, device, batch=4096):
    """全量前向: 窗口按 batch 现场 gather, 返回 np.float32[N] 预测分数。

    截面注意力需要 sec_ids 才能保持"同截面内交互"语义, 因此这里按 keys
    推导全局截面编号并逐 batch 传入; 未启用 xsec 时 sec_ids=None (等价原逻辑)。
    """
    model.eval()
    n = len(keys)
    preds = np.empty(n, "float32")
    use_xsec = bool(getattr(model, "use_xsec", False))
    sec_ids = _section_ids(keys) if use_xsec else None
    for s in range(0, n, batch):
        idxb = np.arange(s, min(s + batch, n))
        xs = _batch_windows(feats, anchors, idxb, seq_lens, freqs)
        xb = [torch.from_numpy(x).to(device) for x in xs]
        sidb = None if sec_ids is None else \
            torch.from_numpy(sec_ids[idxb]).to(device)
        preds[s:s + len(idxb)] = model(xb, sec_ids=sidb).cpu().numpy().reshape(-1)
    return preds


@torch.no_grad()
def _eval_ic(model, feats, anchors, seq_lens, y, keys, freqs, device, batch=4096):
    """验证集截面 IC: 窗口按 batch 现场 gather, 不预物化 [N, T, F]。"""
    preds = _predict_all(model, feats, anchors, seq_lens, freqs, keys,
                         device, batch=batch)
    m = np.isfinite(y)
    if m.sum() < 10:
        return 0.0
    df = pd.DataFrame({"p": preds[m], "y": y[m], "day": keys[m, 1], "mod": keys[m, 2]})
    ics = []
    for _, g in df.groupby(["day", "mod"], sort=False):
        if len(g) < 10:
            continue
        pr = g["p"].rank(); yr = g["y"].rank()
        ics.append(np.corrcoef(pr, yr)[0, 1])
    return float(np.mean(ics)) if ics else 0.0


@torch.no_grad()
def _eval_turnover(model, feats, anchors, seq_lens, freqs, keys, device,
                   batch=4096):
    """验证集换手指标: 同一只股票相邻截面的因子「排名变化」均值 (越低越好)。

    口径: 对每只股票按 (instrument, day) 分组, 组内按截面时间排序, 相邻截面的
    因子排名之差的绝对值求和取平均 (排名按全局截面内归一化到 [0,1] 后衡量)。
    与平台"换手率=信号变化/调仓成本"方向一致 (低换手更好)。
    """
    preds = _predict_all(model, feats, anchors, seq_lens, freqs, keys,
                         device, batch=batch)
    n = len(keys)
    df = pd.DataFrame({
        "inst": keys[:, 0], "day": keys[:, 1], "mod": keys[:, 2], "p": preds,
    })
    # 截面内百分位 (平均名次), 归一化到 [0, 1]
    df["p_rank"] = df.groupby(["day", "mod"], sort=False)["p"].rank(pct=True)
    total_diff = 0.0
    n_trans = 0
    for _, g in df.groupby(["inst", "day"], sort=False):
        if len(g) < 2:
            continue
        g = g.sort_values("mod")
        d = np.abs(np.diff(g["p_rank"].to_numpy()))
        total_diff += float(d.sum())
        n_trans += len(d)
    return float(total_diff / n_trans) if n_trans else float("nan")


def train_and_save(datasources, train_months=None, freqs=None,
                   model_path=MODEL_PATH, lam_turnover=None):
    """
    从零训练并保存 transformer_model.json(私榜重训亦调用本函数)。

    train_months: 训练月份列表; 默认 config.TRAIN_MONTHS (2020-01 .. 2023-12)。
    freqs:        参与建模的频率; 默认 config.FREQS。
    lam_turnover: 换手正则权重 (覆盖 config.LAMBDA_TURNOVER; None 用 config 值)。

    内存: 数据全程按月流式, 不物化滑窗矩阵; 峰值由 "一个月的 bar1m + 各频率
    特征矩阵" 决定, 远低于平台 51200MB 硬限制。
    """
    if train_months is None:
        train_months = CFG.TRAIN_MONTHS
    if freqs is None:
        freqs = CFG.FREQS
    train_months = sorted(str(m) for m in train_months)

    # 实际生效的换手正则权重 (与 _train_freqs 内部一致, 供 payload 自描述)
    lam_to = (float(getattr(CFG, "LAMBDA_TURNOVER", 0.01))
              if lam_turnover is None else float(lam_turnover))

    table = datasources["bar1m"]
    print(f"[train] 分月流式读取 {len(train_months)} 个月 bar1m ... seed={SEED}")

    # 各频率输入窗口 bar 数 = LOOKBACK_MINUTES / bar 长度
    seq_lens = {f: max(1, CFG.LOOKBACK_MINUTES // FREQ_MINUTES[f]) for f in freqs}
    model, stats, model_cfg = _train_freqs(
        table, train_months, list(freqs), seq_lens, ALIGNED_MINUTES, seed=SEED,
        lam_turnover=lam_turnover)

    payload = {
        "format": "miniquant-transformer-v1",
        "arch": model_cfg["arch"],
        "freqs": list(freqs),
        "seq_lens": {f: int(seq_lens[f]) for f in freqs},
        "model_cfg": model_cfg,
        "mean": {f: stats[f][0] for f in freqs},
        "std": {f: stats[f][1] for f in freqs},
        "features": ALL_FEATURES,
        "aligned_minutes": ALIGNED_MINUTES,
        "train_months": list(train_months),
        "seed": int(SEED),
        "train_cfg": dict(CFG.TRAIN_CFG),
        "use_xsec": bool(getattr(CFG, "USE_XSEC", False)),
        "use_turnover": bool(getattr(CFG, "USE_TURNOVER", False)),
        "lambda_turnover": lam_to,
        "label_direction": "end_over_vwap",  # label = end_price/vwap - 1 (新方向, 不取负)
        "lookback_minutes": int(CFG.LOOKBACK_MINUTES),
        "horizon_minutes": int(CFG.HORIZON_MINUTES),
        "sort_key": 1,           # resample/make_anchors 已按时间排序
        "state_dict": {k: v for k, v in model.state_dict().items()},
    }
    # 融合包: hs_* 高分模型权重是团队预训练的固定件, 本函数只重训自有模型。
    # save_model 会整文件覆盖, 故先取出旧文件里的 hs_* 键, 保存后原样写回,
    # 否则私榜重训会把它们冲掉, main() 随后会抛"融合权重缺少模型键"。
    hs_kept = {}
    if os.path.exists(model_path):
        try:
            with open(model_path, "r", encoding="utf-8") as fh:
                hs_kept = {k: v for k, v in json.load(fh).items()
                           if k.startswith("hs_")}
        except (OSError, ValueError):
            hs_kept = {}
    save_model(model_path, payload)
    if hs_kept:
        with open(model_path, "r", encoding="utf-8") as fh:
            fresh = json.load(fh)
        fresh.update(hs_kept)
        with open(model_path, "w", encoding="utf-8") as fh:
            json.dump(fresh, fh)
        print(f"[train] 已保留高分模型权重键 {sorted(hs_kept)} "
              f"({os.path.getsize(model_path)/1e6:.2f} MB)")
    print(f"[train] 已保存 {model_path} ({os.path.getsize(model_path)/1e6:.2f} MB)")
    return model_path



## 高分模型段


In [ ]:
# ==========================================================================
# 高分模型段: exp033 (30m 单频) + exp034 (30m+5m 跨频)
# --------------------------------------------------------------------------
# 与自有模型逐截面 rank 相关仅 ~0.52 (同族高分模型之间是 0.83), 说明二者
# 捕捉的是不同的 alpha 来源 -> 融合是真正的互补, 而非重复计算。
# 实测 2024-07~12 六个独立月份 6/6 正增益, 平均 IC +11.6%。
#
# 本节由 highscore_merged 的 model.py + predict.py 合并而来, 计算逻辑不变;
# 仅做了三处**机械适配**:
#   1. 去掉 `import model`, 把 model.<符号> 直接展开到本命名空间;
#   2. 重命名 4 个与训练脚本冲突的符号:
#        load_model/save_model -> hs_load_model/hs_save_model, main -> hs_main,
#        N_FEATURES -> HS_N_FEATURES (同名但语义不同);
#   3. 权重路径指向本包的 transformer_model.json (与 ours 权重合并在同一文件)。
# ==========================================================================
from __future__ import annotations

from pathlib import Path
from time import perf_counter
import base64
import hashlib
import os

"""文件用处：exp033 ⊕ exp034-λ0.05 分数层融合提交包的模型定义。

一个 model.py 同时容纳两个已训练结构，权重合并在同一个 model.json 里：
1. `HierarchicalMarketResidualAttention`：exp033 的 30m 单频主干（与 exp022 residual_common 逐位一致）。
2. `CrossFrequencyMarketResidual`：exp034 的 30m 主干 + 5m 高频增量编码器 + 查询驱动跨频注意力。

平台只允许五个文件的包契约，故不能再拆 `model_033.py` / `model_034.py`；两种结构共存于本文件，
`load_models` 负责把 `{"exp033": ..., "exp034": ...}` 的合并权重分别解码。
"""


import base64
import hashlib
import json
from pathlib import Path

import numpy as np
import torch
from torch import nn
import torch.nn.functional as F


MAX_WEIGHT_JSON_BYTES = 50_000_000


# ---------------------------------------------------------------------------
# 30m 主干：与 exp022 residual_common 逐位一致
# ---------------------------------------------------------------------------
class MaskedTemporalConvBlock(nn.Module):
    """仅沿单只股票时间轴卷积的残差块；无效 token 始终在块末清零。"""

    def __init__(self, d_model: int, dropout: float) -> None:
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.conv1 = nn.Conv1d(d_model, d_model, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(d_model, d_model, kernel_size=3, padding=1)
        self.dropout = nn.Dropout(dropout)

    def forward(self, h: torch.Tensor, pad_mask: torch.Tensor) -> torch.Tensor:
        valid = (~pad_mask).unsqueeze(-1)
        y = self.norm(h).masked_fill(~valid, 0.0).transpose(1, 2)
        y = self.conv1(y)
        y = self.dropout(F.gelu(y))
        y = self.conv2(y).transpose(1, 2)
        return (h + self.dropout(y)).masked_fill(~valid, 0.0)


class MaskedStridedConv(nn.Module):
    """mask-aware 的 3×1、stride=2 时序 CNN 下采样，不跨股票混合。"""

    def __init__(self, d_model: int, dropout: float) -> None:
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.conv = nn.Conv1d(d_model, d_model, kernel_size=3, stride=2, padding=1, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, h: torch.Tensor, pad_mask: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        valid = ~pad_mask
        x = self.norm(h).masked_fill(~valid.unsqueeze(-1), 0.0).transpose(1, 2)
        out = self.dropout(F.gelu(self.conv(x))).transpose(1, 2)
        out_valid = F.max_pool1d(valid.to(torch.float32).unsqueeze(1), 3, 2, 1).squeeze(1).bool()
        out_pad = ~out_valid
        return out.masked_fill(out_pad.unsqueeze(-1), 0.0), out_pad


class MarketResidualDecomposition(nn.Module):
    """单个 C 层的公共分支；与 exp022 逐位一致，保证共享主干初始化不变。"""

    def __init__(self, d_model: int, gamma_init: float) -> None:
        super().__init__()
        self.residual_norm = nn.LayerNorm(d_model)
        self.market_norm = nn.LayerNorm(d_model)
        self.market_gate_norm = nn.LayerNorm(d_model)
        self.market_proj = nn.Linear(d_model, d_model, bias=False)
        self.gate_mlp = nn.Sequential(
            nn.Linear(2 * d_model, d_model // 4), nn.GELU(),
            nn.Linear(d_model // 4, 1), nn.Sigmoid(),
        )
        self.gamma = nn.Parameter(torch.tensor(float(gamma_init)))

    def common(self, residual: torch.Tensor, market: torch.Tensor,
               valid: torch.Tensor, mode: str) -> tuple[torch.Tensor, torch.Tensor]:
        """计算标量 gate 与公共分支贡献；mean_gate 仅在有效股票内取均值。"""
        market_broadcast = market.unsqueeze(0).expand(residual.shape[0], -1, -1)
        gate_input = torch.cat((self.residual_norm(residual),
                                self.market_gate_norm(market_broadcast)), dim=-1)
        gate = self.gate_mlp(gate_input).masked_fill(~valid.unsqueeze(-1), 0.0)
        if mode == "mean_gate":
            count = valid.sum(dim=0).clamp_min(1).to(gate.dtype).reshape(1, -1, 1)
            mean_gate = gate.sum(dim=0, keepdim=True) / count
            gate = mean_gate.expand_as(gate).masked_fill(~valid.unsqueeze(-1), 0.0)
        common = self.market_proj(self.market_norm(market)).unsqueeze(0)
        contribution = self.gamma * gate * common
        if mode == "common_off":
            contribution = torch.zeros_like(contribution)
        return gate, contribution.masked_fill(~valid.unsqueeze(-1), 0.0)


class HierarchicalMarketResidualAttention(nn.Module):
    """exp022 的 16→8→4 主干，C1/C2 前做逐 section、逐时间市场—残差分解。"""

    def __init__(self, n_in: int = 27, d_model: int = 192, nhead: int = 8,
                 dim_ff: int = 224, dropout: float = 0.1, activation: str = "gelu",
                 norm_first: bool = True, seq_len: int = 16, n_slots: int = 8,
                 cross_time_chunk: int = 2, gamma_init: float = 0.1,
                 variant: str = "residual_common") -> None:
        super().__init__()
        if seq_len != 16:
            raise ValueError("本模型当前只定义了 16→8→4 的时间金字塔")
        if variant not in {"residual_only", "residual_common"}:
            raise ValueError("variant 必须为 residual_only 或 residual_common")
        self.model_config = {
            "n_in": n_in, "d_model": d_model, "nhead": nhead, "dim_ff": dim_ff,
            "dropout": dropout, "activation": activation, "norm_first": norm_first,
            "seq_len": seq_len, "n_slots": n_slots, "cross_time_chunk": cross_time_chunk,
            "gamma_init": gamma_init, "variant": variant,
        }
        self.seq_len, self.cross_time_chunk, self.variant = seq_len, cross_time_chunk, variant
        # 模块名称、创建顺序和初始化方式与 exp020/exp022 完全一致。
        self.input_proj = nn.Linear(n_in, d_model)
        self.pos16 = nn.Parameter(torch.zeros(1, 16, d_model))
        self.pos8 = nn.Parameter(torch.zeros(1, 8, d_model))
        self.pos4 = nn.Parameter(torch.zeros(1, 4, d_model))

        def attention_layer() -> nn.TransformerEncoderLayer:
            return nn.TransformerEncoderLayer(
                d_model=d_model, nhead=nhead, dim_feedforward=dim_ff, dropout=dropout,
                activation=activation, batch_first=True, norm_first=norm_first,
            )

        self.t1, self.t2, self.t3 = attention_layer(), attention_layer(), attention_layer()
        self.c1, self.c2 = attention_layer(), attention_layer()
        self.conv16 = MaskedTemporalConvBlock(d_model, dropout)
        self.down16_8 = MaskedStridedConv(d_model, dropout)
        self.conv8 = MaskedTemporalConvBlock(d_model, dropout)
        self.down8_4 = MaskedStridedConv(d_model, dropout)
        self.conv4 = MaskedTemporalConvBlock(d_model, dropout)
        self.slot_emb = nn.Embedding(n_slots, d_model)
        self.head = nn.Sequential(
            nn.Linear(3 * d_model, d_model), nn.GELU(), nn.LayerNorm(d_model), nn.Linear(d_model, 1),
        )
        for position in (self.pos16, self.pos8, self.pos4):
            nn.init.normal_(position, mean=0.0, std=0.02)

        if variant == "residual_common":
            self.market_c1 = MarketResidualDecomposition(d_model, gamma_init)
            self.market_c2 = MarketResidualDecomposition(d_model, gamma_init)

    def num_parameters(self) -> int:
        return sum(parameter.numel() for parameter in self.parameters())

    @staticmethod
    def _time(layer: nn.TransformerEncoderLayer, h: torch.Tensor,
              pad_mask: torch.Tensor) -> torch.Tensor:
        return layer(h, src_key_padding_mask=pad_mask).masked_fill(pad_mask.unsqueeze(-1), 0.0)

    def _cross(self, layer: nn.TransformerEncoderLayer, h: torch.Tensor,
               pad_mask: torch.Tensor) -> torch.Tensor:
        """沿股票维运行单个 section 的截面 Attention，并跳过全无效时间 slice。"""
        x, mask = h.transpose(0, 1), pad_mask.transpose(0, 1)
        pieces: list[torch.Tensor] = []
        step = max(1, int(self.cross_time_chunk))
        for lo in range(0, x.shape[0], step):
            xb, mb = x[lo:lo + step], mask[lo:lo + step]
            any_valid = (~mb).any(dim=1)
            if bool(any_valid.all()):
                pieces.append(layer(xb, src_key_padding_mask=mb).masked_fill(mb.unsqueeze(-1), 0.0))
            elif not bool(any_valid.any()):
                pieces.append(torch.zeros_like(xb))
            else:
                part = torch.zeros_like(xb)
                part[any_valid] = layer(xb[any_valid], src_key_padding_mask=mb[any_valid]).masked_fill(
                    mb[any_valid].unsqueeze(-1), 0.0)
                pieces.append(part)
        return torch.cat(pieces, dim=0).transpose(0, 1)

    @staticmethod
    def masked_market_residual(h: torch.Tensor, pad_mask: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        """对单个 section 逐时间求股票 masked mean，并将无效位置严格清零。"""
        valid = ~pad_mask
        count = valid.sum(dim=0).clamp_min(1).to(h.dtype).unsqueeze(-1)
        market = (h * valid.unsqueeze(-1)).sum(dim=0) / count
        market = market.masked_fill(~valid.any(dim=0).unsqueeze(-1), 0.0)
        residual = (h - market.unsqueeze(0)).masked_fill(~valid.unsqueeze(-1), 0.0)
        return market, residual

    def _decomposed_cross(self, layer: nn.TransformerEncoderLayer,
                          common_module: MarketResidualDecomposition | None,
                          h: torch.Tensor, pad_mask: torch.Tensor,
                          section_ids: torch.Tensor, position: torch.Tensor,
                          common_mode: str,
                          collect: bool) -> tuple[torch.Tensor, list[dict[str, torch.Tensor]]]:
        """先分解纯隐表示，再给残差加位置编码并逐 section 做 Attention。"""
        if common_mode not in {"normal", "common_off", "mean_gate"}:
            raise ValueError("common_mode 必须为 normal、common_off 或 mean_gate")
        out = torch.zeros_like(h)
        traces: list[dict[str, torch.Tensor]] = []
        for section in torch.unique(section_ids, sorted=True):
            rows = section_ids == section
            hs, mask = h[rows], pad_mask[rows]
            market, residual = self.masked_market_residual(hs, mask)
            attention_input = self._add_position(residual, position, mask)
            attention = self._cross(layer, attention_input, mask)
            gate = None
            contribution = torch.zeros_like(attention)
            if common_module is not None:
                gate, contribution = common_module.common(residual, market, ~mask, common_mode)
            z = (attention + contribution).masked_fill(mask.unsqueeze(-1), 0.0)
            out[rows] = z
            if collect:
                traces.append({"section": section, "H": hs, "M": market, "R": residual,
                               "attention_input": attention_input,
                               "A": attention, "G": gate, "common": contribution, "Z": z,
                               "pad_mask": mask})
        return out, traces

    @staticmethod
    def _add_position(h: torch.Tensor, position: torch.Tensor,
                      pad_mask: torch.Tensor) -> torch.Tensor:
        return (h + position).masked_fill(pad_mask.unsqueeze(-1), 0.0)

    @staticmethod
    def _latest_and_mean(h: torch.Tensor, pad_mask: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        valid = ~pad_mask
        if not bool(valid.any(dim=1).all()):
            raise ValueError("每只股票至少需要一个有效的下采样时间 token")
        positions = torch.arange(h.shape[1], device=h.device).unsqueeze(0).expand_as(pad_mask)
        latest_index = positions.masked_fill(~valid, -1).max(dim=1).values
        latest = h[torch.arange(h.shape[0], device=h.device), latest_index]
        mean = (h * valid.unsqueeze(-1)).sum(dim=1) / valid.sum(dim=1, keepdim=True).clamp_min(1)
        return latest, mean

    def forward_features(self, x: torch.Tensor, pad_mask: torch.Tensor,
                         section_ids: torch.Tensor | None = None, *,
                         common_mode: str = "normal", collect: bool = False):
        if x.ndim != 3 or x.shape[1:] != (16, self.model_config["n_in"]):
            raise ValueError(f"输入应为 (N,16,{self.model_config['n_in']})，实际 {tuple(x.shape)}")
        if pad_mask.shape != x.shape[:2] or pad_mask.dtype != torch.bool:
            raise ValueError("pad_mask 必须是 (N,16) 的 bool 张量")
        if section_ids is None:
            section_ids = torch.zeros(x.shape[0], dtype=torch.long, device=x.device)
        if section_ids.shape != (x.shape[0],):
            raise ValueError("section_ids 必须为每只股票一个截面标识")
        h = self._add_position(self.input_proj(x), self.pos16, pad_mask)
        h = self._time(self.t1, h, pad_mask)
        h = self.conv16(h, pad_mask)
        h, pad8 = self.down16_8(h, pad_mask)
        h, trace1 = self._decomposed_cross(
            self.c1, getattr(self, "market_c1", None), h, pad8, section_ids,
            self.pos8, common_mode, collect)
        h = self._time(self.t2, h, pad8)
        h = self.conv8(h, pad8)
        h, pad4 = self.down8_4(h, pad8)
        h, trace2 = self._decomposed_cross(
            self.c2, getattr(self, "market_c2", None), h, pad4, section_ids,
            self.pos4, common_mode, collect)
        h = self._time(self.t3, h, pad4)
        h = self.conv4(h, pad4)
        return (h, pad4, {"C1": trace1, "C2": trace2}) if collect else (h, pad4)

    def forward(self, x: torch.Tensor, slot: torch.Tensor, pad_mask: torch.Tensor,
                section_ids: torch.Tensor | None = None, *, common_mode: str = "normal",
                collect: bool = False):
        """exp033 的 30m 单频前向，输出路径与原 exp033 提交包逐位一致。"""
        features = self.forward_features(x, pad_mask, section_ids,
                                         common_mode=common_mode, collect=collect)
        h, pad4 = features[:2]
        latest, mean = self._latest_and_mean(h, pad4)
        if slot.shape != (x.shape[0],):
            raise ValueError("slot 必须为每只股票一个整数")
        prediction = self.head(torch.cat((latest, mean, self.slot_emb(slot)), dim=-1)).squeeze(-1)
        return (prediction, features[2]) if collect else prediction


# ---------------------------------------------------------------------------
# 5m 高频增量编码器
# ---------------------------------------------------------------------------
class SegmentedCausalDepthwiseConv1d(nn.Module):
    """严格左因果的深度可分离时序卷积；segment 改变时历史被截断。"""

    def __init__(self, channels: int, kernel_size: int, *, bias: bool = True) -> None:
        super().__init__()
        if kernel_size < 1:
            raise ValueError("kernel_size 必须为正")
        self.kernel_size = int(kernel_size)
        self.conv = nn.Conv1d(channels, channels, kernel_size, padding=0,
                              groups=channels, bias=bias)

    def forward(self, x: torch.Tensor, valid: torch.Tensor, segment: torch.Tensor) -> torch.Tensor:
        if x.shape[:2] != valid.shape or valid.shape != segment.shape:
            raise ValueError("x/valid/segment 形状不匹配")
        clean = x.masked_fill(~valid.unsqueeze(-1), 0.0).transpose(1, 2)
        padded = F.pad(clean, (self.kernel_size - 1, 0))
        windows = padded.unfold(-1, self.kernel_size, 1)
        shifted = F.pad(segment, (self.kernel_size - 1, 0), value=-1).unfold(1, self.kernel_size, 1)
        use = (shifted == segment.unsqueeze(-1)) & (shifted >= 0)
        weight = self.conv.weight.reshape(1, -1, 1, self.kernel_size)
        value = (windows * use.unsqueeze(1).to(windows.dtype) * weight).sum(-1)
        bias = self.conv.bias
        if bias is None:
            raise RuntimeError("分段深度卷积必须带 bias")
        value = value + bias[None, :, None]
        return value.transpose(1, 2)


class SdpaCrossAttention(nn.Module):
    """不物化完整权重的 SDPA cross attention，并提供小张量手算参考路径。"""

    def __init__(self, d_model: int, heads: int, dropout: float) -> None:
        super().__init__()
        if d_model % heads:
            raise ValueError("d_model 必须能整除 heads")
        self.d_model, self.heads, self.head_dim = d_model, heads, d_model // heads
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.dropout = float(dropout)

    def _project(self, query: torch.Tensor, key_value: torch.Tensor):
        batch_shape = query.shape[:-2]
        b = int(np.prod(batch_shape))
        nq, nk = query.shape[-2], key_value.shape[-2]
        q = self.q_proj(query).reshape(b, nq, self.heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(key_value).reshape(b, nk, self.heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(key_value).reshape(b, nk, self.heads, self.head_dim).transpose(1, 2)
        return batch_shape, b, nq, q, k, v

    def forward(self, query: torch.Tensor, key_value: torch.Tensor, key_valid: torch.Tensor,
                query_valid: torch.Tensor | None = None) -> torch.Tensor:
        batch_shape, b, nq, q, k, v = self._project(query, key_value)
        valid = key_valid.reshape(b, key_valid.shape[-1])
        any_valid = valid.any(dim=-1)
        safe = valid.clone()
        safe[~any_valid, 0] = True
        attended = F.scaled_dot_product_attention(
            q, k, v, attn_mask=safe[:, None, None, :],
            dropout_p=self.dropout if self.training else 0.0,
        ).transpose(1, 2).reshape(b, nq, self.d_model)
        out = self.out_proj(attended).reshape(*batch_shape, nq, self.d_model)
        out = out.masked_fill(~any_valid.reshape(*batch_shape, 1, 1), 0.0)
        if query_valid is not None:
            out = out.masked_fill(~query_valid.unsqueeze(-1), 0.0)
        return out

    def reference(self, query: torch.Tensor, key_value: torch.Tensor, key_valid: torch.Tensor,
                  query_valid: torch.Tensor | None = None) -> torch.Tensor:
        batch_shape, b, nq, q, k, v = self._project(query, key_value)
        valid = key_valid.reshape(b, key_valid.shape[-1])
        any_valid = valid.any(-1)
        safe = valid.clone()
        safe[~any_valid, 0] = True
        score = torch.matmul(q, k.transpose(-2, -1)) / float(self.head_dim) ** 0.5
        score = score.masked_fill(~safe[:, None, None, :], -torch.inf)
        attended = torch.matmul(torch.softmax(score, dim=-1), v)
        attended = attended.transpose(1, 2).reshape(b, nq, self.d_model)
        out = self.out_proj(attended).reshape(*batch_shape, nq, self.d_model)
        out = out.masked_fill(~any_valid.reshape(*batch_shape, 1, 1), 0.0)
        if query_valid is not None:
            out = out.masked_fill(~query_valid.unsqueeze(-1), 0.0)
        return out


class LatentSelfAttentionBlock(nn.Module):
    """逐 (section,time) 的单层 Latent self-attention + FFN。"""

    def __init__(self, d5: int, heads: int, ffn: int, dropout: float) -> None:
        super().__init__()
        self.norm1, self.norm2 = nn.LayerNorm(d5), nn.LayerNorm(d5)
        self.attn = SdpaCrossAttention(d5, heads, dropout)
        self.ffn = nn.Sequential(nn.Linear(d5, ffn), nn.GELU(), nn.Dropout(dropout),
                                 nn.Linear(ffn, d5), nn.Dropout(dropout))

    def forward(self, latent: torch.Tensor, time_valid: torch.Tensor) -> torch.Tensor:
        all_valid = time_valid.unsqueeze(-1).expand(*time_valid.shape, latent.shape[-2])
        normed = self.norm1(latent).masked_fill(~all_valid.unsqueeze(-1), 0.0)
        h = latent + self.attn(normed, normed, all_valid)
        h = h + self.ffn(self.norm2(h))
        return h.masked_fill(~all_valid.unsqueeze(-1), 0.0)


class MaskedBinPool(nn.Module):
    """把 96 个 5m 格点按 16 个 bin（每 bin 6 格）做 masked mean + masked last 池化。"""

    def __init__(self, d5: int) -> None:
        super().__init__()
        self.proj = nn.Linear(2 * d5, d5)

    def forward(self, x: torch.Tensor, valid: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        shape = x.shape[:-2]
        cells = x.reshape(*shape, 16, 6, x.shape[-1])
        mask = valid.reshape(*shape, 16, 6)
        weight = mask.unsqueeze(-1).to(x.dtype)
        denom = mask.sum(-1).clamp_min(1).to(x.dtype)
        mean = (cells * weight).sum(-2) / denom.unsqueeze(-1)
        position = torch.arange(6, device=x.device, dtype=x.dtype)
        latest_at = position.masked_fill(~mask, -1.0).max(-1).values
        safe = latest_at.clamp_min(0).to(torch.long)
        index = safe.unsqueeze(-1).unsqueeze(-1).expand(*shape, 16, 1, x.shape[-1])
        latest = cells.gather(-2, index).squeeze(-2)
        exists = mask.any(-1)
        value = self.proj(torch.cat((latest, mean), -1)).masked_fill(~exists.unsqueeze(-1), 0.0)
        return value, exists


class MaskedSegmentedTemporalConv(nn.Module):
    """96→16 的分段时序卷积（kernel=7、stride=6、右侧补 1），不跨日/午休混合。"""

    def __init__(self, channels: int, kernel_size: int = 7, stride: int = 6,
                 dropout: float = 0.1) -> None:
        super().__init__()
        if kernel_size != stride + 1:
            raise ValueError("本实现固定 kernel=stride+1，保证输出恰好 16 个 bin")
        self.kernel_size, self.stride = int(kernel_size), int(stride)
        self.norm = nn.LayerNorm(channels)
        self.conv = nn.Conv1d(channels, channels, kernel_size, stride=stride,
                              padding=0, bias=True)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, valid: torch.Tensor,
                segment: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        if x.ndim != 3 or x.shape[1] != 96 or valid.shape != (x.shape[0], 96):
            raise ValueError("时序卷积输入必须是 (B,96,C)/(B,96)")
        clean = self.norm(x).masked_fill(~valid.unsqueeze(-1), 0.0).transpose(1, 2)
        padded = F.pad(clean, (0, self.kernel_size - self.stride))
        windows = padded.unfold(-1, self.kernel_size, self.stride)
        shifted = F.pad(segment, (0, self.kernel_size - self.stride), value=-1) \
            .unfold(1, self.kernel_size, self.stride)
        center = segment[:, ::self.stride]
        use = (shifted == center.unsqueeze(-1)) & (shifted >= 0)
        taps = windows * use.unsqueeze(1).to(windows.dtype)
        value = torch.einsum("bctk,ock->bot", taps, self.conv.weight)
        bias = self.conv.bias
        if bias is None:
            raise RuntimeError("分段时序卷积必须带 bias")
        value = F.gelu(value + bias[None, :, None])
        output_valid = valid.reshape(x.shape[0], 16, 6).any(-1)
        value = value.masked_fill(~output_valid.unsqueeze(1), 0.0)
        return self.dropout(value.transpose(1, 2)), output_valid


class FiveMinuteIncrementEncoder(nn.Module):
    """98 槽 → 96 格 → 16 个高频增量 token 的残差/公共编码器。"""

    def __init__(self, d5: int = 128, n_latents: int = 64, heads: int = 4,
                 latent_ffn: int = 192, local_kernel: int = 3, temporal_kernel: int = 7,
                 dropout: float = 0.1, d_model: int = 192,
                 gamma_cross_init: float = 0.1, gamma_common_init: float = 0.1) -> None:
        super().__init__()
        self.d5, self.n_latents = int(d5), int(n_latents)
        self.local_in = nn.Linear(27, d5)
        self.bin_pos = nn.Parameter(torch.zeros(1, 16, 1, d5))
        self.intra_pos = nn.Parameter(torch.zeros(1, 1, 6, d5))
        self.local_norm = nn.LayerNorm(d5)
        self.local_conv = SegmentedCausalDepthwiseConv1d(d5, local_kernel)
        self.latent_tokens = nn.Parameter(torch.empty(n_latents, d5))
        self.stock_latent_attention = SdpaCrossAttention(d5, heads, dropout)
        self.latent_self = LatentSelfAttentionBlock(d5, heads, latent_ffn, dropout)
        self.residual_norm = nn.LayerNorm(d5)
        self.bin_pool = MaskedBinPool(d5)
        self.latent_temporal = MaskedSegmentedTemporalConv(d5, temporal_kernel, 6, dropout)
        self.market_temporal = MaskedSegmentedTemporalConv(d5, temporal_kernel, 6, dropout)
        self.latent_stock_attention = SdpaCrossAttention(d5, heads, dropout)
        self.market_norm = nn.LayerNorm(d5)
        self.market_gate_norm = nn.LayerNorm(d5)
        self.market_proj = nn.Linear(d5, d5, bias=False)
        self.common_gate = nn.Sequential(
            nn.Linear(2 * d5, max(4, d5 // 4)), nn.GELU(),
            nn.Linear(max(4, d5 // 4), 1), nn.Sigmoid(),
        )
        self.gamma_cross = nn.Parameter(torch.tensor(float(gamma_cross_init)))
        self.gamma_common = nn.Parameter(torch.tensor(float(gamma_common_init)))
        self.to_model = nn.Linear(d5, d_model)
        for parameter in (self.bin_pos, self.intra_pos, self.latent_tokens):
            nn.init.normal_(parameter, mean=0.0, std=0.02)

    @staticmethod
    def _market_residual(h: torch.Tensor, valid: torch.Tensor):
        """h (S,R,T,d)、valid (S,R,T)；逐时间按真实截面求市场均值与残差。"""
        count = valid.sum(1).clamp_min(1).to(h.dtype).unsqueeze(-1)
        market = (h * valid.unsqueeze(-1)).sum(1) / count
        market = market.masked_fill(~valid.any(1).unsqueeze(-1), 0.0)
        residual = (h - market.unsqueeze(1)).masked_fill(~valid.unsqueeze(-1), 0.0)
        return market, residual

    @staticmethod
    def _group(h: torch.Tensor, valid: torch.Tensor, segment: torch.Tensor,
               section_ids: torch.Tensor):
        sections = torch.unique(section_ids, sorted=True)
        sizes = [int((section_ids == section).sum()) for section in sections]
        if not sizes or len(set(sizes)) != 1:
            raise ValueError("5m 快路要求每个 section 股票数相同；训练按 pair 两端分别前向")
        grouped_h = torch.stack([h[section_ids == section] for section in sections])
        grouped_valid = torch.stack([valid[section_ids == section] for section in sections])
        grouped_segment = torch.stack([segment[section_ids == section] for section in sections])
        return sections, grouped_h, grouped_valid, grouped_segment

    def forward(self, x5: torch.Tensor, pad5: torch.Tensor, slot: torch.Tensor,
                section_ids: torch.Tensor, gather: torch.Tensor,
                cell_segment: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor, dict]:
        n = x5.shape[0]
        mapping = gather[slot]                                  # (n,16,6)
        flat = mapping.reshape(n, 96)
        safe = flat.clamp_min(0)
        cell_valid = (~pad5).gather(1, safe) & (flat >= 0)      # (n,96)
        cells = x5.gather(1, safe.unsqueeze(-1).expand(-1, -1, x5.shape[-1]))
        cells = cells * cell_valid.unsqueeze(-1).to(cells.dtype)
        segment = cell_segment[slot].reshape(n, 96)

        h = self.local_in(cells).reshape(n, 16, 6, self.d5)
        h = h + self.bin_pos.reshape(1, 16, 1, -1) + self.intra_pos.reshape(1, 1, 6, -1)
        h = h.reshape(n, 96, self.d5)
        h = h + self.local_conv(self.local_norm(h), cell_valid, segment)
        h = h.masked_fill(~cell_valid.unsqueeze(-1), 0.0)

        sections, gh, gv, gs = self._group(h, cell_valid, segment, section_ids)
        rows_per = gh.shape[1]
        market, residual = self._market_residual(gh, gv)

        residual_st = residual.permute(0, 2, 1, 3)              # (S,96,R,d5)
        valid_st = gv.permute(0, 2, 1)                          # (S,96,R)
        latent = self.stock_latent_attention(
            self.latent_tokens.reshape(1, 1, -1, self.d5).expand(residual_st.shape[0], 96, -1, -1),
            self.residual_norm(residual_st), valid_st)          # (S,96,L,d5)
        latent = self.latent_self(latent, valid_st.any(-1))

        section_count, latent_count = latent.shape[0], latent.shape[2]
        cell_any = gv.any(1)                                    # (S,96)
        seg_section = gs[:, 0]                                  # (S,96)
        latent_in = latent.permute(0, 2, 1, 3).reshape(section_count * latent_count, 96, self.d5)
        latent_valid = cell_any.unsqueeze(1).expand(section_count, latent_count, 96) \
            .reshape(section_count * latent_count, 96)
        latent_segment = seg_section.unsqueeze(1).expand(section_count, latent_count, 96) \
            .reshape(section_count * latent_count, 96)
        latent16, bin_valid = self.latent_temporal(latent_in, latent_valid, latent_segment)
        latent16 = latent16.reshape(section_count, latent_count, 16, self.d5).permute(0, 2, 1, 3)
        bin_valid = bin_valid.reshape(section_count, latent_count, 16).permute(0, 2, 1)

        market16, _ = self.market_temporal(market, cell_any, seg_section)   # (S,16,d5)

        r16, rv16 = self.bin_pool(residual.reshape(section_count * rows_per, 96, self.d5),
                                  gv.reshape(section_count * rows_per, 96))
        r16 = r16.reshape(section_count, rows_per, 16, self.d5).permute(0, 2, 1, 3)
        rv16 = rv16.reshape(section_count, rows_per, 16).permute(0, 2, 1)

        attention = self.latent_stock_attention(r16, latent16, bin_valid, rv16)
        market_broadcast = market16.unsqueeze(2).expand(-1, -1, rows_per, -1)
        common = self.market_proj(self.market_norm(market_broadcast))
        gate = self.common_gate(torch.cat((self.residual_norm(r16),
                                           self.market_gate_norm(market_broadcast)), -1))
        common_valid = rv16 & bin_valid.any(-1).unsqueeze(2)
        gate = gate.masked_fill(~common_valid.unsqueeze(-1), 0.0)
        attention = attention.masked_fill(~rv16.unsqueeze(-1), 0.0)
        contribution = (self.gamma_common * gate * common) \
            .masked_fill(~common_valid.unsqueeze(-1), 0.0)
        z = (r16 + self.gamma_cross * attention + contribution) \
            .masked_fill(~rv16.unsqueeze(-1), 0.0)

        z_flat = z.permute(0, 2, 1, 3).reshape(section_count * rows_per, 16, self.d5)
        valid_flat = rv16.permute(0, 2, 1).reshape(section_count * rows_per, 16)
        order = torch.cat([torch.nonzero(section_ids == section, as_tuple=False).squeeze(1)
                           for section in sections])
        inverse = torch.empty_like(order)
        inverse[order] = torch.arange(len(order), device=order.device)
        tokens = self.to_model(z_flat[inverse])
        pad16 = ~valid_flat[inverse]
        info = {"cell_valid": cell_valid, "cell_segment": segment, "R5": residual,
                "latent": latent, "latent16": latent16, "bin_valid": bin_valid,
                "R16": r16, "M16": market16, "A16": attention, "G16": gate,
                "common16": contribution, "tokens16": tokens}
        return tokens.masked_fill(pad16.unsqueeze(-1), 0.0), pad16, info


class QueryGuidedCrossFrequency(nn.Module):
    """4 个 30m 表征 token 作为 query，主动查询 16 个 5m 增量 token，门控残差回注。"""

    def __init__(self, d_model: int, heads: int, dropout: float, gamma_init: float) -> None:
        super().__init__()
        self.q_norm, self.kv_norm = nn.LayerNorm(d_model), nn.LayerNorm(d_model)
        self.attn = SdpaCrossAttention(d_model, heads, dropout)
        self.gate = nn.Sequential(
            nn.Linear(2 * d_model, d_model // 4), nn.GELU(),
            nn.Linear(d_model // 4, 1), nn.Sigmoid(),
        )
        self.gamma = nn.Parameter(torch.tensor(float(gamma_init)))

    def forward(self, query: torch.Tensor, query_pad: torch.Tensor,
                key_value: torch.Tensor, key_pad: torch.Tensor) -> tuple[torch.Tensor, dict]:
        query_valid = ~query_pad
        attention = self.attn(self.q_norm(query), self.kv_norm(key_value), ~key_pad, query_valid)
        gate = self.gate(torch.cat((self.q_norm(query), attention), -1)) \
            .masked_fill(~query_valid.unsqueeze(-1), 0.0)
        increment = self.gamma * gate * attention
        return query + increment, {"attention": attention, "gate": gate, "increment": increment}


class CrossFrequencyMarketResidual(HierarchicalMarketResidualAttention):
    """exp022 主干 + 5m 增量编码器 + 查询驱动跨频门控残差，最后接原预测 Head。"""

    def __init__(self, *, d5: int = 128, n_latents: int = 64, hf_heads: int = 4,
                 latent_ffn: int = 192, local_kernel: int = 3, temporal_kernel: int = 7,
                 freq_heads: int = 8, freq_dropout: float = 0.1,
                 gamma_cross_init: float = 0.1, gamma_common_init: float = 0.1,
                 gamma_freq_init: float = 0.1, **kwargs) -> None:
        kwargs.setdefault("variant", "residual_common")
        if kwargs["variant"] != "residual_common":
            raise ValueError("exp032 只定义 residual_common 主干")
        super().__init__(**kwargs)
        self.model_config = {
            **self.model_config, "d5": d5, "n_latents": n_latents, "hf_heads": hf_heads,
            "latent_ffn": latent_ffn, "local_kernel": local_kernel,
            "temporal_kernel": temporal_kernel, "freq_heads": freq_heads,
            "freq_dropout": freq_dropout, "gamma_cross_init": gamma_cross_init,
            "gamma_common_init": gamma_common_init, "gamma_freq_init": gamma_freq_init,
        }
        # 新模块必须晚于完整 exp022 主干创建，避免改变共享参数的随机初始化序列。
        self.five = FiveMinuteIncrementEncoder(
            d5=d5, n_latents=n_latents, heads=hf_heads, latent_ffn=latent_ffn,
            local_kernel=local_kernel, temporal_kernel=temporal_kernel,
            dropout=self.model_config["dropout"], d_model=self.model_config["d_model"],
            gamma_cross_init=gamma_cross_init, gamma_common_init=gamma_common_init)
        self.freq = QueryGuidedCrossFrequency(
            self.model_config["d_model"], freq_heads, freq_dropout, gamma_freq_init)

    def forward(self, x: torch.Tensor, slot: torch.Tensor, pad_mask: torch.Tensor,
                section_ids: torch.Tensor | None = None, *, x5: torch.Tensor | None = None,
                pad5: torch.Tensor | None = None, gather: torch.Tensor | None = None,
                cell_segment: torch.Tensor | None = None, common_mode: str = "normal",
                freq_off: bool = False, collect: bool = False):
        features = self.forward_features(x, pad_mask, section_ids,
                                         common_mode=common_mode, collect=collect)
        h, pad4 = features[:2]
        freq_trace = None
        if not freq_off:
            if x5 is None or pad5 is None or gather is None or cell_segment is None:
                raise ValueError("未关闭跨频分支时必须提供 x5/pad5/gather/cell_segment")
            if section_ids is None:
                section_ids = torch.zeros(x.shape[0], dtype=torch.long, device=x.device)
            tokens16, pad16, five_trace = self.five(
                x5, pad5, slot, section_ids, gather, cell_segment)
            h, freq_trace = self.freq(h, pad4, tokens16, pad16)
            freq_trace = {**freq_trace, "five": five_trace, "pad16": pad16}
        latest, mean = self._latest_and_mean(h, pad4)
        if slot.shape != (x.shape[0],):
            raise ValueError("slot 必须为每只股票一个整数")
        prediction = self.head(torch.cat((latest, mean, self.slot_emb(slot)), dim=-1)).squeeze(-1)
        if collect:
            traces = features[2]
            traces["freq"] = freq_trace
            return prediction, traces
        return prediction


def hs_save_model(payload: dict, path: Path) -> Path:
    """把 checkpoint 张量无损编码进文本 JSON，并硬性限制文件小于 50 MB。"""
    def convert(value):
        if isinstance(value, torch.Tensor):
            cpu = value.detach().cpu().contiguous()
            return {"dtype": str(cpu.dtype).removeprefix("torch."), "shape": list(cpu.shape),
                    "encoding": "base64_raw", "data": base64.b64encode(cpu.numpy().tobytes()).decode("ascii")}
        if isinstance(value, dict):
            return {key: convert(item) for key, item in value.items()}
        if isinstance(value, (list, tuple)):
            return [convert(item) for item in value]
        return value
    document = json.dumps(convert(payload), ensure_ascii=False, separators=(",", ":"))
    size = len(document.encode("utf-8"))
    if size >= MAX_WEIGHT_JSON_BYTES:
        raise RuntimeError(f"JSON 权重 {size} bytes，不满足 <{MAX_WEIGHT_JSON_BYTES} bytes 硬限制")
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(document, encoding="utf-8")
    return path


def decode_payload(payload: dict, map_location: str = "cpu") -> dict:
    """把单个已编码的 JSON payload 还原为张量 state_dict。"""
    state = {}
    for name, meta in payload["state_dict"].items():
        dtype = getattr(torch, meta["dtype"])
        if meta.get("encoding") == "base64_raw":
            array = np.frombuffer(base64.b64decode(meta["data"]), dtype=meta["dtype"]).copy()
            state[name] = torch.from_numpy(array).to(dtype=dtype).reshape(meta["shape"]).to(map_location)
        else:
            state[name] = torch.tensor(meta["data"], dtype=dtype).reshape(meta["shape"]).to(map_location)
    return {**{key: value for key, value in payload.items() if key != "state_dict"}, "state_dict": state}


def hs_load_model(path: str | Path, map_location: str = "cpu") -> dict:
    """读取单模型 JSON 文本权重并还原 state_dict 张量。"""
    return decode_payload(json.loads(Path(path).read_text(encoding="utf-8")), map_location)


def load_models(path: str | Path, keys: tuple[str, ...], map_location: str = "cpu") -> dict[str, dict]:
    """读取融合包合并权重：按 exp033 / exp034 两个键分别解码，缺键直接报错。"""
    document = json.loads(Path(path).read_text(encoding="utf-8"))
    missing = [key for key in keys if key not in document]
    if missing:
        raise RuntimeError(f"融合权重缺少模型键：{missing}")
    return {key: decode_payload(document[key], map_location) for key in keys}


def build_model_from_cfg(model_cfg: dict) -> CrossFrequencyMarketResidual:
    """按 JSON 中保存的固定配置重建 exp034 跨频模型。"""
    return CrossFrequencyMarketResidual(**model_cfg)


def build_thirty_minute_from_cfg(model_cfg: dict) -> HierarchicalMarketResidualAttention:
    """按 JSON 中保存的固定配置重建 exp033 30m 单频模型。"""
    return HierarchicalMarketResidualAttention(**model_cfg)


"""文件用处：平台入口——同一次行情上分别跑 exp033 与 exp034-λ0.05，再按截面 rank 等权（0.5:0.5）融合。

融合口径与 `score-level-fusion-protocol` 一致：
1. 两个模型各自输出原始分数（量纲相差约 40 倍，不能直接加权）；
2. 每个截面对各自分数取 `rank(pct=True)`；
3. 融合分数 = 0.5 * rank(exp033) + 0.5 * rank(exp034)；
4. 只在一个模型有样本时退化为该模型的 rank（可用案例等权），保证覆盖率不被弱模型拖低。

两路输入共享同一份 30m 特征（两个权重里的 27 维 mean/std 逐位相同），5m 只服务 exp034。
"""


from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
import torch


ROOT = Path(os.path.abspath("."))
SECTION_TABLE = "cpt_jyc_2026_instruments"
DECISION_HHMM = (930, 1000, 1030, 1100, 1130, 1330, 1400, 1430)
FUSION_WEIGHT_033 = 0.5
FUSION_WEIGHT_034 = 0.5

# 原始字段顺序必须与 exp033 features.py 的 RAW_COLS 一致（位置索引即特征索引）。
RAW_FIELDS = (
    "open", "high", "low", "close", "volume", "amount", "deal_number",
    "ask_price1", "ask_price2", "ask_price3",
    "bid_price1", "bid_price2", "bid_price3",
    "ask_volume1", "ask_volume2", "ask_volume3",
    "bid_volume1", "bid_volume2", "bid_volume3",
    "ask_num_orders1", "ask_num_orders2", "ask_num_orders3",
    "bid_num_orders1", "bid_num_orders2", "bid_num_orders3",
)
C_VOLUME = 4
HS_N_FEATURES, N_CHANNEL, CH_BAR_VALID, CH_GAP_FLAG = 25, 27, 25, 26
# 09:25 竞价空 bar 需要置 NaN 的 15 个量类通道
QUANTITY_COLS = np.asarray([4, 5, 6, *range(13, 25)], dtype=np.int64)
# 盘口 18 列与 (价, 量, 笔数) 三元组
OB_COLS = list(range(7, 25))
OB_GROUPS = [(7 + level, 13 + level, 19 + level) for level in range(3)] + [
    (10 + level, 16 + level, 22 + level) for level in range(3)
]
COL_ASK1, COL_BID1 = 7, 10

WINDOW, MIN_VALID_BARS, N_DAY_SLOTS, LEAD_IN_DAYS = 16, 8, 9, 40
N_FINE, N_FINE_DAY = 98, 49
CLIP_Z = 50.0
GRID30 = np.asarray([565, 600, 630, 660, 690, 810, 840, 870, 900], dtype=np.int64)
INPUT30 = np.asarray([565, 600, 630, 660, 690, 810, 840, 870], dtype=np.int64)
GRID5 = np.asarray([565, *range(575, 691, 5), *range(785, 901, 5)], dtype=np.int64)

# exp033 cache_5m/stats_5m.npz 的前 25 通道训练集统计量（float64 -> float32 存储）。
# 权重 JSON 只携带 30m 的 27 维 mean/std；受平台五个文件的包契约限制，5m 统计量内嵌在此。
STATS_5M_MEAN = np.asarray([
    -1.0228791325062048e-05, 0.0025131588336080313, -0.0024907742626965046,
    8.036107828957029e-06, 11.730358123779297, 14.321381568908691, 5.214993000030518,
    0.0007045346428640187, 0.001939702546223998, 0.0031563700176775455,
    -0.0006759742391295731, -0.0018791864858940244, -0.0030583813786506653,
    8.788694381713867, 9.130496978759766, 9.138110160827637, 8.910387992858887,
    9.378867149353027, 9.418570518493652, 1.880773663520813, 2.0349669456481934,
    2.009232521057129, 2.0223891735076904, 2.2782208919525146, 2.2873470783233643,
], dtype=np.float64)
STATS_5M_STD = np.asarray([
    0.004427162930369377, 0.003119669621810317, 0.0034492728300392628,
    0.005138415843248367, 1.456773281097412, 1.379389762878418, 1.1471514701843262,
    0.001048273523338139, 0.0016601962270215154, 0.00244133360683918,
    0.0010221891570836306, 0.0016215358627960086, 0.002404178725555539,
    2.25759220123291, 2.3025965690612793, 2.2873685359954834, 2.3412699699401855,
    2.326787233352661, 2.287137031555176, 1.1234970092773438, 1.1155669689178467,
    1.094411015510559, 1.228183627128601, 1.2129415273666382, 1.1887296438217163,
], dtype=np.float64)


def validate_datasources(datasources: dict) -> None:
    """拒绝缺少任一频率的数据源，防止双频模型被静默降级为单频。"""
    if not isinstance(datasources, dict) or {"bar30m", "bar5m"} - set(datasources):
        raise KeyError("datasources 必须同时含 bar30m 与 bar5m")


def _query(sql: str, filters: dict) -> pd.DataFrame:
    import dai
    try:
        return dai.query(sql, filters=filters, compression=True).df()
    except TypeError:
        return dai.query(sql, filters=filters).df()


def _sections(start: str, end: str) -> pd.DataFrame:
    """读取成分股截面表，并映射到 0..7 的决策槽（09:30..14:30）。"""
    data = _query(f"SELECT date, instrument::string AS instrument FROM {SECTION_TABLE}",
                  {"date": [start, end]})
    data["date"] = pd.to_datetime(data["date"])
    data = data.dropna(subset=["date", "instrument"]).drop_duplicates(["date", "instrument"])
    data = data[(data.date >= pd.Timestamp(start)) & (data.date < pd.Timestamp(end))].copy()
    hhmm = data.date.dt.hour * 100 + data.date.dt.minute
    if data.empty or not hhmm.isin(DECISION_HHMM).all():
        raise RuntimeError("成分股表为空或含非法截面时刻")
    data["slot"] = np.searchsorted(np.asarray(DECISION_HHMM, dtype=np.int64), hhmm.to_numpy())
    return data


def _bars(table: str, start: str, end: str, pool: list[str]) -> pd.DataFrame:
    """读取行情并按 (instrument, date) 稳定排序；30m 与 5m 表结构一致，共用本函数。"""
    columns = ", ".join(["date", "instrument::string AS instrument", "adjust_factor", *RAW_FIELDS])
    data = _query(f"SELECT {columns} FROM {table}", {"date": [start, end], "instrument": pool})
    data["date"] = pd.to_datetime(data["date"])
    return data.dropna(subset=["date", "instrument"]).drop_duplicates(
        ["instrument", "date"]).sort_values(["instrument", "date"], kind="stable").reset_index(drop=True)


def _lattice(frame: pd.DataFrame, days: np.ndarray, grid: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """把单只股票行情映射到「交易日下标 × 时间格点」，返回 (交易日下标, 格点下标, 分钟)。"""
    day_index = {pd.Timestamp(day): i for i, day in enumerate(days)}
    day = frame.date.dt.normalize().map(day_index).to_numpy()
    minute = (frame.date.dt.hour * 60 + frame.date.dt.minute).to_numpy(np.int64)
    slot = np.searchsorted(grid, minute).astype(np.int64)
    inside = (slot < len(grid)) & (grid[np.clip(slot, 0, len(grid) - 1)] == minute)
    if np.isnan(day).any() or not inside.all():
        raise RuntimeError("行情包含不在训练格点内的时间")
    return day.astype(np.int64), slot, minute


def _mask_order_book(raw: np.ndarray) -> None:
    """exp033 features.mask_order_book：盘口 0 价与倒挂掩码，必须在任何比值/对数变换之前做。"""
    for price_col, volume_col, order_col in OB_GROUPS:
        empty = raw[:, price_col] == 0
        if empty.any():
            raw[empty, price_col] = np.nan
            raw[empty, volume_col] = np.nan
            raw[empty, order_col] = np.nan
    ask1, bid1 = raw[:, COL_ASK1], raw[:, COL_BID1]
    crossed = np.isfinite(ask1) & np.isfinite(bid1) & (ask1 < bid1)
    if crossed.any():
        raw[np.ix_(np.nonzero(crossed)[0], OB_COLS)] = np.nan


def _compute_features(raw: np.ndarray, close_adj: np.ndarray, prev_close_adj: np.ndarray) -> np.ndarray:
    """exp033 features.compute_features：25 个 example 式特征，(n, 25) float64，缺失保持 NaN。"""
    feature = np.full((raw.shape[0], HS_N_FEATURES), np.nan, dtype=np.float64)
    close = raw[:, 3]
    close_ok = np.isfinite(close) & (close > 0)
    denom = np.where(close_ok, close, 1.0)
    for index in (0, 1, 2):
        value = raw[:, index]
        feature[:, index] = np.where(close_ok & np.isfinite(value), value / denom - 1.0, np.nan)
    prev_ok = np.isfinite(prev_close_adj) & (prev_close_adj > 0)
    feature[:, 3] = np.where(close_ok & prev_ok,
                             close_adj / np.where(prev_ok, prev_close_adj, 1.0) - 1.0, np.nan)
    for index in (4, 5, 6):
        value = raw[:, index]
        feature[:, index] = np.where(np.isfinite(value), np.log1p(np.clip(value, 0.0, None)), np.nan)
    for index in range(7, 13):
        value = raw[:, index]
        feature[:, index] = np.where(close_ok & np.isfinite(value), value / denom - 1.0, np.nan)
    for index in range(13, 25):
        value = raw[:, index]
        feature[:, index] = np.where(np.isfinite(value), np.log1p(np.clip(value, 0.0, None)), np.nan)
    return feature


def _bar_valid(raw: np.ndarray, slot: np.ndarray) -> np.ndarray:
    """bar_valid := OHLC 有限 且 (volume > 0 或 slot == 0)；slot0 是 09:25 竞价 bar，只看价。"""
    finite_ohlc = np.isfinite(raw[:, :4]).all(axis=1)
    return finite_ohlc & ((raw[:, C_VOLUME] > 0) | (slot == 0))


def _previous_finite(values: np.ndarray) -> np.ndarray:
    """逐格给出同股上一条有限且为正的值（跳过缺失格），用于复权 close_ret。"""
    previous = np.full(values.shape, np.nan, dtype=np.float64)
    last = np.nan
    for index, value in enumerate(values):
        previous[index] = last
        if np.isfinite(value) and value > 0:
            last = value
    return previous


def _features30(bars: pd.DataFrame, mean: np.ndarray, std: np.ndarray,
                days: np.ndarray) -> dict[str, tuple[np.ndarray, int]]:
    """按 exp033 训练缓存口径重建 30m 27 通道，返回 {股票: (通道矩阵, 首个 lattice 下标)}。"""
    result: dict[str, tuple[np.ndarray, int]] = {}
    for code, frame in bars.groupby("instrument", sort=False):
        day, slot, _ = _lattice(frame, days, GRID30)
        lattice = day * N_DAY_SLOTS + slot
        first = int(lattice[0])
        local = lattice - first
        length = int(local[-1]) + 1
        dense = np.full((length, 26), np.nan, dtype=np.float64)
        dense[local, :HS_N_FEATURES] = frame[list(RAW_FIELDS)].to_numpy(np.float64)
        dense[local, HS_N_FEATURES] = frame["adjust_factor"].to_numpy(np.float64)
        _mask_order_book(dense)
        close = dense[:, 3]
        close_ok = np.isfinite(close) & (close > 0)
        close_adj = np.where(close_ok, close * dense[:, HS_N_FEATURES], np.nan)
        feature = _compute_features(dense, close_adj, _previous_finite(close_adj))
        slot_all = (first + np.arange(length)) % N_DAY_SLOTS
        valid = _bar_valid(dense, slot_all)
        gap = np.zeros(length, dtype=np.float64)
        gap[local[1:]] = (np.diff(local) != 1).astype(np.float64)
        # 09:25 竞价空 bar：只把 15 个量类特征置 NaN（z-score 后为 0），价类保留。
        auction_empty = (slot_all == 0) & ~(dense[:, C_VOLUME] > 0)
        if auction_empty.any():
            feature[np.ix_(np.nonzero(auction_empty)[0], QUANTITY_COLS)] = np.nan
        z = (feature - mean[:HS_N_FEATURES]) / std[:HS_N_FEATURES]
        z[~np.isfinite(z)] = 0.0
        stacked = np.concatenate((z, valid[:, None].astype(np.float64), gap[:, None]), axis=1)
        result[str(code)] = (stacked.astype(np.float32), first)
    return result


def _features5(bars: pd.DataFrame, days: np.ndarray) -> dict[str, np.ndarray]:
    """按 exp033 build_data_5m 口径重建 5m 27 通道（未标准化），返回 {股票: (交易日, 49, 27)}。"""
    result: dict[str, np.ndarray] = {}
    last_close: dict[str, float] = {}
    for code, frame in bars.groupby("instrument", sort=False):
        day, _, minute = _lattice(frame, days, GRID5)
        values = frame[[*RAW_FIELDS, "adjust_factor"]].to_numpy(np.float64)
        raw, adjust = values[:, :HS_N_FEATURES], values[:, HS_N_FEATURES]
        close = raw[:, 3]
        close_ok = np.isfinite(close) & (close > 0)
        close_adj = np.where(close_ok, close * adjust, np.nan)
        # 与构建脚本一致：日内用「上一行」的复权收盘（不跳过无效行），跨日接该股上一交易日的最后有效收盘。
        previous = np.full(close_adj.shape, np.nan, dtype=np.float64)
        same_day = day[1:] == day[:-1]
        previous[1:][same_day] = close_adj[:-1][same_day]
        starts = np.flatnonzero(np.concatenate(([True], day[1:] != day[:-1])))
        previous[starts] = last_close.get(str(code), np.nan)
        _mask_order_book(raw)
        feature = _compute_features(raw, close_adj, previous)
        auction = minute == 565
        valid = _bar_valid(raw, np.where(auction, 0, 1))
        auction_empty = auction & ~(raw[:, C_VOLUME] > 0)
        if auction_empty.any():
            feature[np.ix_(np.nonzero(auction_empty)[0], QUANTITY_COLS)] = np.nan
        block = np.zeros((len(days), N_FINE_DAY, N_CHANNEL), dtype=np.float64)
        local = np.searchsorted(GRID5, minute)
        block[day, local, :HS_N_FEATURES] = feature
        block[day, local, CH_BAR_VALID] = valid.astype(np.float64)
        for current in np.unique(day):
            closes = close_adj[day == current]
            closes = closes[np.isfinite(closes) & (closes > 0)]
            if closes.size:
                last_close[str(code)] = float(closes[-1])
        result[str(code)] = block.astype(np.float32)
    return result


def _cell_tables() -> tuple[np.ndarray, np.ndarray]:
    """按 exp034 alignment.py 重建 8×16×6 的真实 5m 格点映射（每 30m token 最多六格）。"""
    bin_index = np.full((8, N_FINE), -1, dtype=np.int8)
    segment5 = np.asarray([
        day * 3 + (0 if minute == 565 else (1 if minute <= 690 else 2))
        for day in range(2) for minute in GRID5
    ], dtype=np.int8)
    for slot, input_end in enumerate(INPUT30):
        endpoint_index = int(np.flatnonzero(GRID30 == input_end)[0])
        previous_start = 6 - endpoint_index
        current_start = 15 - endpoint_index
        endpoint_to_bin = {(0, int(endpoint)): previous_start + index
                           for index, endpoint in enumerate(GRID30)}
        endpoint_to_bin.update({(1, int(GRID30[index])): current_start + index
                                for index in range(endpoint_index + 1)})
        for day in range(2):
            for local_index, minute in enumerate(GRID5):
                if day == 1 and minute > input_end:
                    continue
                endpoint = 565 if minute == 565 else next(
                    int(end) for end in GRID30[1:] if end - 25 <= minute <= end)
                target = endpoint_to_bin.get((day, endpoint), -1)
                if 0 <= target < 16:
                    bin_index[slot, day * len(GRID5) + local_index] = target
    gather = np.full((8, 16, 6), -1, dtype=np.int16)
    cell_segment = np.full((8, 16, 6), -1, dtype=np.int8)
    counts = np.zeros((8, 16), dtype=np.int16)
    for slot in range(8):
        for source in range(N_FINE):
            target = int(bin_index[slot, source])
            if target < 0:
                continue
            position = int(counts[slot, target])
            if position >= 6:
                raise RuntimeError("5m 格点映射超过每个 30m token 的六格契约")
            gather[slot, target, position] = source
            cell_segment[slot, target, position] = segment5[source]
            counts[slot, target] += 1
    return gather, cell_segment


def _samples30(section: pd.DataFrame, features30: dict[str, tuple[np.ndarray, int]],
               day_index: dict[pd.Timestamp, int], pad30: np.ndarray
               ) -> tuple[np.ndarray, np.ndarray, list[str]]:
    """exp033 单频样本：只切 (16,27) 的 30m 窗口，不要求 5m 覆盖。"""
    day = pd.Timestamp(section.date.iloc[0]).normalize()
    current = day_index.get(day)
    if current is None:
        return np.empty((0, WINDOW, N_CHANNEL), np.float32), np.empty(0, np.int64), []
    decision_slot = int(section.slot.iloc[0])
    windows, slots, codes = [], [], []
    for row in section.itertuples(index=False):
        coarse = features30.get(row.instrument)
        if coarse is None:
            continue
        flat30, first = coarse
        end30 = current * N_DAY_SLOTS + decision_slot - first
        if end30 < 0 or end30 >= flat30.shape[0]:
            continue
        # 09:30 以 09:25 竞价 bar 为窗口末位（允许无成交）；其余截面必须有当期有效 30m bar。
        if decision_slot > 0 and flat30[end30, CH_BAR_VALID] <= 0:
            continue
        padding = max(0, WINDOW - 1 - end30)
        window30 = flat30[end30 - (WINDOW - 1 - padding):end30 + 1]
        if padding:
            window30 = np.concatenate((np.tile(pad30, (padding, 1)), window30), axis=0)
        if int((window30[:, CH_BAR_VALID] > 0).sum()) < MIN_VALID_BARS:
            continue
        windows.append(window30.astype(np.float32))
        slots.append(decision_slot)
        codes.append(row.instrument)
    if not codes:
        return np.empty((0, WINDOW, N_CHANNEL), np.float32), np.empty(0, np.int64), []
    return np.asarray(windows, np.float32), np.asarray(slots, np.int64), codes


def _samples(section: pd.DataFrame, features30: dict[str, tuple[np.ndarray, int]],
             features5: dict[str, np.ndarray], days: np.ndarray,
             mean5: np.ndarray, std5: np.ndarray, pad30: np.ndarray
             ) -> tuple[np.ndarray, np.ndarray, np.ndarray, list[str]]:
    """逐股票切出 (16,27) 的 30m 窗口与 (98,27) 的 5m 窗口，口径与训练样本一致。"""
    day_index = {pd.Timestamp(day): i for i, day in enumerate(days)}
    x30, x5, slots, codes = [], [], [], []
    day = pd.Timestamp(section.date.iloc[0]).normalize()
    current = day_index.get(day)
    if current is None:
        return (np.empty((0, WINDOW, N_CHANNEL), np.float32), np.empty((0, N_FINE, N_CHANNEL), np.float32),
                np.empty(0, np.int64), [])
    decision_slot = int(section.slot.iloc[0])
    future5 = N_FINE_DAY + np.flatnonzero(GRID5 > INPUT30[decision_slot])
    for row in section.itertuples(index=False):
        coarse = features30.get(row.instrument)
        fine = features5.get(row.instrument)
        if coarse is None or fine is None:
            continue
        flat30, first = coarse
        end30 = current * len(GRID30) + decision_slot - first
        if end30 < 0 or end30 >= flat30.shape[0]:
            continue
        # 09:30 以 09:25 竞价 bar 为窗口末位（允许无成交）；其余截面必须有当期有效 30m bar。
        if decision_slot > 0 and flat30[end30, CH_BAR_VALID] <= 0:
            continue
        padding = max(0, WINDOW - 1 - end30)
        window30 = flat30[end30 - (WINDOW - 1 - padding):end30 + 1]
        if padding:
            window30 = np.concatenate((np.tile(pad30, (padding, 1)), window30), axis=0)
        if int((window30[:, CH_BAR_VALID] > 0).sum()) < MIN_VALID_BARS:
            continue
        previous = fine[current - 1] if current >= 1 else np.zeros_like(fine[0])
        window5 = np.concatenate((previous, fine[current]), axis=0).copy()
        # 与缓存一致：先把未写入/未来格置 0，再标准化，这些格因此为 -mean/std 且 bar_valid=0。
        window5[future5] = 0.0
        z5 = (window5[:, :HS_N_FEATURES] - mean5[:HS_N_FEATURES]) / std5[:HS_N_FEATURES]
        z5[~np.isfinite(z5)] = 0.0
        np.clip(z5, -CLIP_Z, CLIP_Z, out=z5)
        window5 = np.concatenate((z5, window5[:, HS_N_FEATURES:]), axis=1).astype(np.float32)
        x30.append(window30.astype(np.float32))
        x5.append(window5)
        slots.append(decision_slot)
        codes.append(row.instrument)
    if not codes:
        return (np.empty((0, WINDOW, N_CHANNEL), np.float32), np.empty((0, N_FINE, N_CHANNEL), np.float32),
                np.empty(0, np.int64), [])
    return (np.asarray(x30, np.float32), np.asarray(x5, np.float32),
            np.asarray(slots, np.int64), codes)


def _predict_single_frequency(network: torch.nn.Module, sections: pd.DataFrame,
                              features30: dict[str, tuple[np.ndarray, int]], days: np.ndarray,
                              pad30: np.ndarray, device: torch.device) -> pd.DataFrame:
    """exp033 30m 单频逐截面推理，返回原始分数（未做 rank）。"""
    day_index = {pd.Timestamp(day): i for i, day in enumerate(days)}
    frames: list[pd.DataFrame] = []
    for date, section in sections.groupby("date", sort=True):
        x30, slots, codes = _samples30(section, features30, day_index, pad30)
        if not len(codes):
            continue
        xb = torch.from_numpy(x30).to(device)
        slot = torch.from_numpy(slots).to(device)
        with torch.no_grad():
            factor = network(xb, slot, ~(xb[:, :, CH_BAR_VALID] > 0),
                             torch.zeros(len(codes), dtype=torch.long, device=device)
                             ).float().cpu().numpy()
        frames.append(pd.DataFrame({"date": date, "instrument": codes, "factor": factor}))
    return _concat_frames(frames, ("date", "instrument", "factor"))


def _predict_dual_frequency(network: torch.nn.Module, sections: pd.DataFrame,
                            features30: dict[str, tuple[np.ndarray, int]],
                            features5: dict[str, np.ndarray], days: np.ndarray,
                            pad30: np.ndarray, gather: np.ndarray, cell_segment: np.ndarray,
                            device: torch.device) -> pd.DataFrame:
    """exp034 双频逐截面推理，返回原始分数（未做 rank）。"""
    frames: list[pd.DataFrame] = []
    gather_t = torch.from_numpy(gather).to(device, dtype=torch.long)
    segment_t = torch.from_numpy(cell_segment).to(device, dtype=torch.long)
    for date, section in sections.groupby("date", sort=True):
        x30, x5, slots, codes = _samples(section, features30, features5, days,
                                         STATS_5M_MEAN, STATS_5M_STD, pad30)
        if not len(codes):
            continue
        tx30, tx5 = torch.from_numpy(x30).to(device), torch.from_numpy(x5).to(device)
        with torch.no_grad():
            factor = network(
                tx30, torch.from_numpy(slots).to(device), ~(tx30[:, :, CH_BAR_VALID] > 0),
                torch.zeros(len(codes), dtype=torch.long, device=device), x5=tx5,
                pad5=~(tx5[:, :, CH_BAR_VALID] > 0),
                gather=gather_t, cell_segment=segment_t,
            ).float().cpu().numpy()
        frames.append(pd.DataFrame({"date": date, "instrument": codes, "factor": factor}))
    return _concat_frames(frames, ("date", "instrument", "factor"))


def _concat_frames(frames: list[pd.DataFrame], columns: tuple[str, ...]) -> pd.DataFrame:
    """拼接逐截面结果并清洗非有限值；空结果返回同 schema 的空表。"""
    if not frames:
        return pd.DataFrame(columns=list(columns))
    return (pd.concat(frames, ignore_index=True)
            .replace([np.inf, -np.inf], np.nan).dropna(subset=["factor"])
            .drop_duplicates(["date", "instrument"])
            .sort_values(["date", "instrument"], kind="stable").reset_index(drop=True))


def fuse_rank_predictions(primary: pd.DataFrame, secondary: pd.DataFrame,
                          weight: float = FUSION_WEIGHT_033) -> pd.DataFrame:
    """按截面 rank(pct) 等权融合两模型原始分数；单边缺失时退化为可用模型的 rank。

    秩变换是必需的：exp033 原始分数量级约 3.8、exp034 约 0.1，直接加权等于只用其中一个。
    rank 在 (date, instrument) 并集上计算；pandas 的 rank 忽略 NaN，因此等价于各自可用集合内排名。
    """
    if not 0.0 <= weight <= 1.0:
        raise ValueError("融合权重必须位于 [0, 1]")
    if primary.empty:
        return secondary.copy()
    if secondary.empty:
        return primary.copy()
    left = primary[["date", "instrument", "factor"]].rename(columns={"factor": "primary"})
    right = secondary[["date", "instrument", "factor"]].rename(columns={"factor": "secondary"})
    merged = left.merge(right, on=["date", "instrument"], how="outer")
    rank_primary = merged.groupby("date", sort=False)["primary"].rank(pct=True, method="average")
    rank_secondary = merged.groupby("date", sort=False)["secondary"].rank(pct=True, method="average")
    fused = weight * rank_primary + (1.0 - weight) * rank_secondary
    only_primary = fused.isna() & rank_primary.notna()
    only_secondary = fused.isna() & rank_secondary.notna()
    fused = fused.mask(only_primary, rank_primary).mask(only_secondary, rank_secondary)
    merged["factor"] = fused.astype(np.float64)
    return (merged[["date", "instrument", "factor"]]
            .replace([np.inf, -np.inf], np.nan).dropna(subset=["factor"])
            .sort_values(["date", "instrument"], kind="stable").reset_index(drop=True))


def _fill_weak_sections(output: pd.DataFrame, sections: pd.DataFrame) -> pd.DataFrame:
    """覆盖率低于 60% 的截面用该截面中位因子补齐，避免平台硬校验直接判废。"""
    expected = sections.groupby("date").instrument.nunique()
    ratio = output.groupby("date").instrument.nunique().reindex(expected.index, fill_value=0) / expected
    weak = ratio.index[ratio < 0.6]
    if not len(weak):
        return output
    median = output.groupby("date").factor.median().reindex(weak).fillna(0.0)
    pad = sections.loc[sections.date.isin(weak), ["date", "instrument"]].copy()
    pad["factor"] = pad["date"].map(median).astype(float)
    print(f"[warn] {len(weak)} 个截面覆盖率低于 60%，按截面中位因子补齐 {len(pad):,} 行", flush=True)
    return pd.concat((output, pad), ignore_index=True).drop_duplicates(["date", "instrument"], keep="first")


def validate_output(output: pd.DataFrame, sections: pd.DataFrame,
                    start: pd.Timestamp, end: pd.Timestamp) -> None:
    """平台写盘前的五项输出自检（列名/时刻/覆盖率/有限性/唯一性/时间窗）。"""
    if set(output.columns) != {"date", "instrument", "factor"}:
        raise RuntimeError("输出列必须恰为 date/instrument/factor")
    legal = pd.MultiIndex.from_frame(sections[["date", "instrument"]])
    actual = pd.MultiIndex.from_frame(output[["date", "instrument"]])
    hhmm = output.date.dt.hour * 100 + output.date.dt.minute
    if (not np.isfinite(output.factor).all() or output.duplicated(["date", "instrument"]).any()
            or not actual.isin(legal).all() or not hhmm.isin(DECISION_HHMM).all()
            or not ((output.date >= start) & (output.date < end)).all()):
        raise RuntimeError("输出不满足平台日期、有限性或唯一性契约")
    expected = sections.groupby("date").instrument.nunique()
    coverage = output.groupby("date").instrument.nunique().reindex(expected.index, fill_value=0) / expected
    if (coverage < 0.6).any():
        raise RuntimeError(f"截面覆盖率不足 60%，最小值 {coverage.min():.3f}")


def _load_payloads(device: torch.device) -> tuple[dict, dict]:
    """读取合并权重并按模型键校验 30m 元数据。"""
    payloads = load_models(ROOT / "transformer_model.json", ("hs_exp033", "hs_exp034"), str(device))
    for key in ("hs_exp033", "hs_exp034"):
        payload = payloads[key]
        for field in ("model_cfg", "mean", "std", "state_dict"):
            if field not in payload:
                raise RuntimeError(f"融合权重 {key} 缺少 {field}")
        mean = np.asarray(payload["mean"], dtype=np.float64)
        std = np.asarray(payload["std"], dtype=np.float64)
        if int(payload["model_cfg"].get("n_in", 0)) != N_CHANNEL or mean.shape != (N_CHANNEL,) \
                or std.shape != (N_CHANNEL,) or not np.isfinite(std).all() or not (std > 0).all():
            raise RuntimeError(f"融合权重 {key} 的 30m 元数据必须是 27 维有限正标准差")
    return payloads["hs_exp033"], payloads["hs_exp034"]


def hs_main(datasources, start_date, end_date):
    """平台入口：一次行情、两模型推理、截面 rank 等权融合，返回 date/instrument/factor 三列。"""
    started = perf_counter()
    validate_datasources(datasources)
    start = pd.Timestamp(start_date)
    end = pd.Timestamp(end_date).normalize() + pd.Timedelta(days=1)
    sections = _sections(start.strftime("%Y-%m-%d"), end.strftime("%Y-%m-%d"))
    pool = sorted(sections.instrument.unique())
    history_start = (start.normalize() - pd.Timedelta(days=LEAD_IN_DAYS)).strftime("%Y-%m-%d")
    bars30 = _bars(datasources["bar30m"], history_start, end.strftime("%Y-%m-%d"), pool)
    bars5 = _bars(datasources["bar5m"], history_start, end.strftime("%Y-%m-%d"), pool)
    days = np.unique(np.concatenate((sections.date.dt.normalize().to_numpy("datetime64[ns]"),
                                     bars30.date.dt.normalize().to_numpy("datetime64[ns]"),
                                     bars5.date.dt.normalize().to_numpy("datetime64[ns]"))))
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    payload33, payload34 = _load_payloads(device)
    mean33 = np.asarray(payload33["mean"], dtype=np.float64)
    std33 = np.asarray(payload33["std"], dtype=np.float64)
    mean34 = np.asarray(payload34["mean"], dtype=np.float64)
    std34 = np.asarray(payload34["std"], dtype=np.float64)
    pad30_33 = np.zeros((N_CHANNEL,), dtype=np.float32)
    pad30_34 = np.concatenate((-mean34[:HS_N_FEATURES] / std34[:HS_N_FEATURES], np.zeros(2))).astype(np.float32)

    features30_33 = _features30(bars30, mean33, std33, days)
    # 两个权重共享同一份 exp033 30m 缓存，mean/std 逐位相同，此时不重复建特征。
    features30_34 = features30_33 if (np.array_equal(mean33, mean34) and np.array_equal(std33, std34)) \
        else _features30(bars30, mean34, std34, days)
    features5 = _features5(bars5, days)

    network33 = build_thirty_minute_from_cfg(payload33["model_cfg"]).to(device)
    network33.load_state_dict(payload33["state_dict"])
    network33.eval()
    network34 = build_model_from_cfg(payload34["model_cfg"]).to(device)
    network34.load_state_dict(payload34["state_dict"])
    network34.eval()
    gather, cell_segment = _cell_tables()

    prediction33 = _predict_single_frequency(network33, sections, features30_33, days, pad30_33, device)
    prediction34 = _predict_dual_frequency(network34, sections, features30_34, features5, days,
                                           pad30_34, gather, cell_segment, device)
    output = fuse_rank_predictions(prediction33, prediction34, FUSION_WEIGHT_033)
    output = _fill_weak_sections(output, sections)
    validate_output(output, sections, start, end)
    print(f"[{ROOT.name}] exp033 {len(prediction33):,} 行 + exp034 {len(prediction34):,} 行 "
          f"→ 融合 {len(output):,} 行，w={FUSION_WEIGHT_033:.2f}，耗时 {perf_counter() - started:.1f}s",
          flush=True)
    return output[["date", "instrument", "factor"]]


## 融合入口


In [ ]:


# ==========================================================================
# 推理入口: 自有模型 + 高分模型融合 + 日内因果 L1 软阈值降换手
# --------------------------------------------------------------------------
# 融合必要性: 两路信号逐截面 rank 相关仅 ~0.52, 远低于同族模型间的 0.83,
# 说明二者捕捉的是不同的 alpha 来源 -> 融合是互补而非重复。
#
# 融合口径 (与 highscore 的 score-level-fusion-protocol 一致):
#   1. 两路各自输出原始分数, 量纲相差约 40 倍, **不能直接加权**;
#   2. 逐截面取 rank(pct=True) 抹平量纲;
#   3. 融合分 = FUSE_W * rank(ours) + (1-FUSE_W) * rank(hs);
#   4. 只有一路有样本时退化为该路的 rank, 不让弱侧拖低覆盖率。
#
# FUSE_W 落在 IC 曲线的平台区 (0.375~0.5), 不取尖峰值以降低过拟合风险。
#
# 软阈值动机: 换手指标 mean|Δpct_rank| 是 L1 型度量, 因此用同型的 L1 软阈值
# 而非 EMA(L2)。EMA 对所有变化按固定比例衰减, 会把真实的大幅跳变一并糊掉,
# 导致信号滞后、IC 永久损失。等换手预算下 soft 的 IC_IR 损失约为 EMA 的一半
# (实测 -4.7% vs -10.2%) —— 抖动被吸收而真实跳变主体保留。
# ==========================================================================
FUSE_W = 0.4
SOFT_THETA = 0.2

# dai / structlog 在训练脚本里是**函数内局部 import** (第 427 行), 而融合入口
# 在模块级用 both, 因此显式在此导入。
import dai
import structlog

logger = structlog.get_logger()


def _soft_within_day(df, factor, theta):
    """日内因果 L1 软阈值 (按 instrument + 交易日分组, 日内重置)。

    先在**每个截面上做 z-score** (逐截面单调变换, 不改变截面内排名,
    对 IC/IC_IR/Sharpe 口径无副作用), 使 Δ 有统一尺度; 再对偏离施加软阈值:

        p'_t = p'_{t-1} + sign(d) * max(0, |d| - theta),  d = z_t - p'_{t-1}

    |d| <= theta 的抖动被完全吸收 (换手 -> 0);
    |d| >  theta 的跳变只削去 theta 的边缘, 主体保留 (IC 基本无损)。
    因果: p'_t 只依赖当前与历史截面; 日内重置: 每日从首个截面重新起算。
    """
    if theta is None or theta <= 0.0 or len(df) == 0:
        return factor
    d = pd.DataFrame({
        "_i": np.arange(len(df)),
        "inst": df["instrument"].to_numpy(),
        "day": pd.to_datetime(df["date"]).dt.normalize().to_numpy(),
        "t": pd.to_datetime(df["date"]).to_numpy(),
        "f": np.asarray(factor, dtype="float64"),
    })
    # 列轴必须用「日内分钟数」而非完整时间戳: 否则同一列名混杂多天,
    # 每个 (day, inst) 行只有部分列有值, 透视后产生整片 NaN。
    d["tod"] = (pd.to_datetime(d["t"]).dt.hour * 60
                + pd.to_datetime(d["t"]).dt.minute).astype("int64")
    g = d.groupby(["day", "tod"], sort=False, observed=True)["f"]
    mu = g.transform("mean")
    sd = g.transform("std")
    d["z"] = ((d["f"] - mu) / sd.where(sd > 1e-12, 1.0)).fillna(0.0)

    piv = d.pivot_table(index=["day", "inst"], columns="tod", values="z",
                        aggfunc="first", sort=True)
    M = piv.to_numpy(dtype="float64", copy=True)
    if M.shape[1] > 1:
        # 状态机: prev=NaN 表示该行尚未开始 (股票当日还没出现)。首次出现以
        # 当期值为起点; 出现前保持 NaN (不可用未来值回填, 否则构成前视偏差);
        # 中途缺失 (停牌) 沿用上一期。
        prev = np.full(M.shape[0], np.nan)
        for j in range(M.shape[1]):
            cur = M[:, j]
            start = np.isnan(prev) & ~np.isnan(cur)
            dl = np.where(start, 0.0, cur - prev)
            nxt = prev + np.sign(dl) * np.maximum(0.0, np.abs(dl) - theta)
            nxt = np.where(start, cur, nxt)
            nxt = np.where(np.isnan(cur), prev, nxt)
            M[:, j] = nxt
            prev = nxt

    di = piv.index.get_level_values(0)
    ii = piv.index.get_level_values(1)
    ti = piv.columns.to_numpy()
    gi, mi = np.meshgrid(np.arange(len(piv.index)), np.arange(len(ti)),
                         indexing="ij")
    flat = pd.DataFrame({
        "day": di.to_numpy()[gi.ravel()],
        "inst": ii.to_numpy()[gi.ravel()],
        "tod": ti[mi.ravel()].astype("int64"),
        "sm": M.ravel(),
    })
    o = d.merge(flat, on=["day", "inst", "tod"], how="left")
    out = np.empty(len(df), dtype="float64")
    out[o["_i"].to_numpy()] = o["sm"].to_numpy()
    return out


def _fuse_predictions(a, b, w):
    """逐截面 rank(pct) 加权融合; 单侧缺失时退化为可用侧的 rank。"""
    if a.empty:
        return b.copy()
    if b.empty:
        return a.copy()
    m = (a[["date", "instrument", "factor"]].rename(columns={"factor": "pa"})
         .merge(b[["date", "instrument", "factor"]]
                .rename(columns={"factor": "pb"}),
                on=["date", "instrument"], how="outer"))
    ra = m.groupby("date", sort=False)["pa"].rank(pct=True, method="average")
    rb = m.groupby("date", sort=False)["pb"].rank(pct=True, method="average")
    f = w * ra + (1.0 - w) * rb
    # 只有一路有样本时, 上式任一侧为 NaN; 退化为可用侧标量值。
    f = f.mask(f.isna() & ra.notna(), ra).mask(f.isna() & rb.notna(), rb)
    m["factor"] = f.astype(np.float64)
    return (m[["date", "instrument", "factor"]]
            .replace([np.inf, -np.inf], np.nan).dropna(subset=["factor"])
            .sort_values(["date", "instrument"], kind="stable")
            .reset_index(drop=True))


def ours_main(datasources, start_date, end_date):
    """自有模型推理: 加载 transformer_model.json 的 ours 部分并逐月预测。"""
    table = datasources["bar1m"]
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if not os.path.exists(MODEL_PATH):
        raise FileNotFoundError(f"未找到模型文件 {MODEL_PATH}")

    ckpt = load_model(MODEL_PATH, map_location=device)
    freqs = ckpt["freqs"]
    seq_lens = ckpt["seq_lens"]
    stats = {f: (np.asarray(ckpt["mean"][f], np.float32),
                 np.asarray(ckpt["std"][f], np.float32)) for f in freqs}
    model = build_model(ckpt["model_cfg"]).to(device)
    model.load_state_dict(ckpt["state_dict"])
    model.eval()
    use_xsec = bool(ckpt.get("model_cfg", {}).get("use_xsec", False))
    # 方向标记: label_direction="vwap_over_end" 是旧方向权重, 需取负对齐平台口径。
    factor_sign = -1.0 if ckpt.get("label_direction") == "vwap_over_end" else 1.0
    logger.info("已加载自有模型", freqs=freqs, device=str(device),
                use_xsec=use_xsec, factor_sign=factor_sign)

    parts = []
    with torch.no_grad():
        for Xs, _, idx_df, keys in _iter_dataset(
                table, start_date, end_date, "infer",
                ckpt["aligned_minutes"], stats, freqs=freqs, seq_lens=seq_lens):
            if len(idx_df) == 0:
                continue
            Xt = [torch.from_numpy(np.ascontiguousarray(x)) for x in Xs]
            preds = []
            sec_ids = _section_ids(keys) if use_xsec else None
            for i in range(0, len(idx_df), BATCH):
                xb = [t[i:i + BATCH].to(device) for t in Xt]
                sidb = (None if sec_ids is None else
                        torch.from_numpy(sec_ids[i:i + BATCH]).to(device))
                preds.append(model(xb, sec_ids=sidb).cpu().numpy())
            out = idx_df.copy()
            out["factor"] = ((np.concatenate(preds).astype(np.float64) * factor_sign)
                             if preds else np.zeros(len(out)))
            parts.append(out)
            del Xs, Xt, preds, idx_df
            gc.collect()
    pred_df = (pd.concat(parts, ignore_index=True) if parts else
               pd.DataFrame({"date": [], "instrument": [], "factor": []}))
    del parts
    gc.collect()
    logger.info("自有模型推理完成", rows=len(pred_df))

    # 内部 instrument 是 _encode_instrument 的稳定 int 码, 需对齐成平台字符串代码。
    stk = dai.query("SELECT date, instrument::string AS instrument "
                    "FROM cpt_jyc_2026_instruments",
                    filters={"date": [start_date, end_date]}).df()
    pred_df = pred_df.rename(columns={"instrument": "_code"})
    pred_df["date"] = pd.to_datetime(pred_df["date"])
    stk["date"] = pd.to_datetime(stk["date"])
    stk["_code"] = _encode_instrument(stk["instrument"])
    return (pd.merge(stk, pred_df, on=["date", "_code"], how="inner")
              .replace([np.inf, -np.inf], np.nan).dropna(subset=["factor"])
              .drop_duplicates(["date", "instrument"])
              [["date", "instrument", "factor"]]
              .sort_values(["date", "instrument"])
              .reset_index(drop=True))


def main(datasources, start_date, end_date):
    """平台入口: 两路推理 -> 截面 rank 加权融合 -> 日内 L1 软阈值 -> 三列输出。

    本函数**不训练**: ours 权重与 hs 权重 (hs_exp033/hs_exp034) 都在随包上传的
    transformer_model.json 里。start_date~end_date 为平台注入的测试区间,
    输出 ['date','instrument','factor']。
    """
    ours = ours_main(datasources, start_date, end_date)
    hs = hs_main(datasources, start_date, end_date)
    logger.info("两路推理完成", ours=len(ours), hs=len(hs))

    # 两路输出都已按 cpt_jyc_2026_instruments 对齐为平台字符串代码, 可直接 join。
    ours["instrument"] = ours["instrument"].astype(str)
    hs["instrument"] = hs["instrument"].astype(str)

    out = _fuse_predictions(ours, hs, FUSE_W)
    logger.info("融合完成", rows=len(out), w=FUSE_W)

    if SOFT_THETA:
        out["factor"] = _soft_within_day(out, out["factor"].to_numpy(), SOFT_THETA)
        logger.info("软阈值后处理完成", theta=SOFT_THETA)

    logger.info("因子构建完成", rows=len(out), dates=out["date"].nunique(),
                instruments=out["instrument"].nunique())
    return out[["date", "instrument", "factor"]]


In [ ]:
# ============================================================
# 本地 / 演示入口
# 平台公榜会直接调用 main(datasources, start_date, end_date);
# 私榜会调用 train_and_save(datasources) 在隔离环境从零重训。
# 因此正常提交无需运行本单元格 (平台自己调用函数)。
# ============================================================
RUN_LOCAL_DEMO = False          # 本地想看效果改成 True
if RUN_LOCAL_DEMO:
    from bigmodule import M
    datasources = {"bar1m": "cpt_jyc_2026_e2e_bar1m"}
    if not os.path.exists(MODEL_PATH):
        train_and_save(datasources)
    factor_data = main(datasources, "2024-07-01 00:00:00", "2024-09-30 23:59:59")
    print(factor_data.head())
    M.jyc_eval._latest(factor_data=factor_data, show=True)
